# Recommender Systems — Week 2 Study Notebook (v2)
### Baselines → Neighborhood CF (kNN) → Matrix Factorization & LightGCN

**How this notebook is organised.** There are three parts, one per notebook of the professor (`baselines.ipynb`, `knn_cf.ipynb`, `gnn.ipynb`). Every part has the same five sections:

| Section | What you get | Why it is there |
|---|---|---|
| **A. The real-world problem** | A concrete situation from a real product | So you know *why* the method exists |
| **B. Solving it by hand** | The method worked out step by step on one tiny table, with every calculation written out, then the general algorithm | So you can reproduce it on paper in the exam |
| **C. The professor's code** | What his code does, line by line, and why it is written in a way that is fine for teaching but not used in industry | So you can read and answer questions about his code |
| **D. The industry version** | The same job done the way it is done at work, on real data | So you learn the practical way |
| **E. Exam questions** | The professor's own questions plus likely variations, with model answers | Exam preparation |

**One tiny table is used for all hand calculations** (4 users × 5 movies = 20 cells). Every number computed by hand is then checked by a code cell, so you can trust the numbers.

---
# 0. The setting

## 0.1 The story we will follow

Imagine you are the first machine-learning engineer at a small movie-streaming company. Call it **StreamBox**. The product manager comes to you with three requests, in this order:

1. *"Our home page is empty for new users. What do we show them?"* → **Part 1, Baselines**
2. *"I want a row called 'Because you watched X'. And please stop recommending action movies to people who hate them."* → **Part 2, Neighborhood CF**
3. *"Most users never rate anything, they just watch. And the catalog is growing to millions of titles. Make it work."* → **Part 3, MF and LightGCN**

These are the same three steps almost every real recommender team goes through, and they are the three topics of this week.

## 0.2 The tiny dataset

Four users, five movies. A number is the star rating (1–5) the user gave. `?` means the user has not rated the movie. This is called the **user–item rating matrix**.

| | Toy Story | Nemo | Matrix | Terminator | Titanic |
|---|---|---|---|---|---|
| **Ana** | 5 | 4 | 1 | ? | ? |
| **Ben** | 1 | ? | 5 | 4 | ? |
| **Cem** | 4 | 5 | ? | 2 | 3 |
| **Dee** | ? | 2 | 4 | 5 | 1 |

Read the table as a person would:

- **Ana** loves the two animation movies (5, 4) and hates Matrix (1). She looks like an "animation, not action" person.
- **Ben** is the opposite: Toy Story 1, Matrix 5, Terminator 4.
- **Cem** is like Ana (animation high, Terminator low).
- **Dee** is like Ben (action high, Nemo low).

There are 14 known ratings and 6 empty cells. **The whole subject is about the empty cells.**

**The question we will carry through the whole notebook:**

> **Should StreamBox recommend *Terminator* to Ana?**

Your own intuition says *no*: Ana hates Matrix, and the people who like Terminator (Ben, Dee) are the people who like Matrix. Keep this in mind. We will see which methods get this right and which get it wrong, and why.

## 0.3 Two different tasks

Everything this week belongs to one of two tasks. Always ask yourself first which one a question is about, because the methods and the metrics are different.

| | Rating Prediction | Top-N Recommendation |
|---|---|---|
| Question | "What rating would Ana give Terminator?" | "Which N movies should Ana see on her home page?" |
| Output | one number, e.g. 1.9 | a ranked list of N items |
| Type of problem | regression | ranking |
| Data needed | explicit ratings (stars) | explicit ratings **or** implicit behaviour (clicks, views, purchases) |
| How we measure | MAE, RMSE (lower is better) | Hit Ratio, Precision@K, Recall@K, NDCG@K (higher is better) |
| Use in industry | rare today | almost every real product |

**Why is rating prediction rare in industry?** Because users do not see predicted ratings, they see a list. A model can be good at guessing "3.4 vs 3.6 stars" for mediocre movies and still be bad at putting the best five movies at the top. Rating prediction became famous through the Netflix Prize (2006–2009), and it is still taught first because it is the easiest way to understand the ideas.

### Setup

In [1]:
# Colab: run this line once (about 1 minute on Colab)
# %pip install -q scikit-surprise implicit

In [2]:
import os, math, time, random, heapq, bisect, itertools, operator, zipfile, urllib.request, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 4)
SEED = 42
np.random.seed(SEED); random.seed(SEED)

In [3]:
# The tiny table used for every hand calculation
nan = np.nan
T = pd.DataFrame([[5, 4, 1, nan, nan],
                  [1, nan, 5, 4, nan],
                  [4, 5, nan, 2, 3],
                  [nan, 2, 4, 5, 1]],
                 index=["Ana", "Ben", "Cem", "Dee"],
                 columns=["ToyStory", "Nemo", "Matrix", "Terminator", "Titanic"])
T

,ToyStory,Nemo,Matrix,Terminator,Titanic
Ana,5.0,4.0,1.0,NaN,NaN
Ben,1.0,NaN,5.0,4.0,NaN
Cem,4.0,5.0,NaN,2.0,3.0
Dee,NaN,2.0,4.0,5.0,1.0


The real dataset used in sections C and D is **MovieLens (ml-latest-small)**, the same one the professor uses: about 100,000 ratings from 610 users on 9,724 movies.

In [4]:
# MovieLens ml-latest-small: the same dataset the professor uses
if not os.path.exists("ml-latest-small/ratings.csv"):
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", "ml.zip")
    zipfile.ZipFile("ml.zip").extractall()

ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies  = pd.read_csv("ml-latest-small/movies.csv")
title   = movies.set_index("movieId").title

n_users, n_items, n_ratings = ratings.userId.nunique(), ratings.movieId.nunique(), len(ratings)
print(f"users={n_users}  items={n_items}  ratings={n_ratings}")
print(f"density = {n_ratings/(n_users*n_items):.4f}   -> {100*(1-n_ratings/(n_users*n_items)):.1f}% of the matrix is empty")
ratings.head(3)

users=610  items=9724  ratings=100836
density = 0.0170   -> 98.3% of the matrix is empty


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224


**Exam note.** In his code the professor prints `n_ratings/(n_users*n_items)` under the name *Sparsity*. That number (0.017) is actually the **density**: the share of cells that are filled. **Sparsity = 1 − density ≈ 98.3%**, the share of cells that are empty. If the exam asks for the sparsity of this dataset, the answer is about 98.3%.

Two more facts about real data that explain many results later:

In [5]:
item_cnt = ratings.groupby("movieId").size().sort_values(ascending=False)
print("rating distribution:\n", ratings.rating.value_counts(normalize=True).sort_index().round(3).to_string())
print(f"\nitems with <=2 ratings : {100*(item_cnt<=2).mean():.1f}% of the catalog   (long tail)")
print(f"top 10% of items hold  : {100*item_cnt.head(n_items//10).sum()/n_ratings:.1f}% of all ratings (popularity bias)")

rating distribution:
 rating
0.5    0.014
1.0    0.028
1.5    0.018
2.0    0.075
2.5    0.055
3.0    0.199
3.5    0.130
4.0    0.266
4.5    0.085
5.0    0.131

items with <=2 ratings : 48.8% of the catalog   (long tail)
top 10% of items hold  : 60.0% of all ratings (popularity bias)


- **Long tail:** about half of the movies have only 1 or 2 ratings. For those movies we know almost nothing.
- **Popularity bias:** the top 10% of movies receive about 60% of all ratings.

Remember these two. They explain why the popularity baseline is so strong (Part 1), why similarities between rare items are unreliable (Part 2), and why evaluation numbers can be misleading.

---
# Part 1 — Baselines

## 1.A The real-world problem

**Situation.** StreamBox launches tomorrow. You have a rating table but no model. Three things must work on day one:

1. **A new user opens the app.** We know nothing about them. What goes on the home page? (This is called the **cold-start** problem.)
2. **A movie page shows "users rate this 4.2 ★".** Which number do we show, and is it fair for a movie with only two ratings?
3. **Next month you will build a "smart" model.** How will you prove to your manager that it is better than doing nothing clever at all?

A **baseline** is the answer to all three: the simplest method that gives a reasonable result with no learning.

**Why baselines matter at work (not only in class):**

- **They are the yardstick.** "MAE = 0.75" means nothing alone. "MAE = 0.75, and always predicting the average gives 0.84" means something. A complex model that cannot beat a simple baseline is not deployed.
- **They are the cold-start solution.** "Trending now" and "Most popular" rows on Netflix, YouTube or Amazon *are* baselines, running in production.
- **They are the fallback.** When the smart model has no answer (new item, new user, service timeout), the system falls back to a baseline. The professor's code does the same with `.get(movie_id, avg_rating)`.
- **They are features.** In real ranking models, "item popularity" and "item average rating" are fed into the model as input features.

## 1.B Solving it by hand

We answer the running question, *"what would Ana rate Terminator?"*, with four baselines, from the simplest to the best. For each one: how to compute it, why it makes sense, and where it is used in real systems.

### Baseline 1 — the global mean $\mu$

**Idea.** Ignore who the user is and what the movie is. Predict the average of all ratings.

**By hand.** Add all 14 known ratings, row by row:

- Ana: $5+4+1 = 10$
- Ben: $1+5+4 = 10$
- Cem: $4+5+2+3 = 14$
- Dee: $2+4+5+1 = 12$

$$\mu = \frac{10+10+14+12}{14} = \frac{46}{14} = 3.29$$

**Prediction:** Ana → Terminator = **3.29**. (It predicts 3.29 for every user and every movie.)

**Why it is not useless.** If you must predict one number for everything, the mean is the number that makes the squared error as small as possible. So it is the best "know-nothing" guess.

**Where the global mean is used in practice:**

- **Last-resort fallback** when both the user and the item are new.
- **Centering.** Almost every model first subtracts $\mu$ and learns only the deviation from it. You will see $\mu$ inside the bias baseline below and inside matrix-factorization models.
- **The prior in damped averages** (see Top-N baselines below): a movie with few ratings is pulled toward $\mu$.

### Baseline 2 — the user mean $\bar r_u$

**Idea.** Some people are generous and give 4–5 to everything; some are strict. Predict each user's own average.

**By hand.**

| User | Ratings | Sum | Count | $\bar r_u$ |
|---|---|---|---|---|
| Ana | 5, 4, 1 | 10 | 3 | 3.33 |
| Ben | 1, 5, 4 | 10 | 3 | 3.33 |
| Cem | 4, 5, 2, 3 | 14 | 4 | 3.50 |
| Dee | 2, 4, 5, 1 | 12 | 4 | 3.00 |

**Prediction:** Ana → Terminator = Ana's mean = **3.33**.

**What it captures and what it misses.** It captures the user's personal *scale*. It misses the movie completely: it predicts 3.33 for every movie Ana has not seen, so it **cannot rank movies for a user**, and therefore it is useless for Top-N.

**Where the user mean is used in practice:**

- **Normalising users.** A "4" from a strict user means more than a "4" from someone who gives 5 to everything. Subtracting the user's mean puts everybody on the same scale. This is exactly the **mean-centering** step used in adjusted cosine similarity in Part 2, where it turns out to be essential.
- **Fallback for a brand-new item** that has no ratings yet (we know the user, not the item).

### Baseline 3 — the item mean $\bar r_i$

**Idea.** Some movies are simply better liked than others. Predict the movie's average rating.

**By hand** (column by column):

| Movie | Ratings | Sum | Count | $\bar r_i$ |
|---|---|---|---|---|
| Toy Story | 5, 1, 4 | 10 | 3 | 3.33 |
| Nemo | 4, 5, 2 | 11 | 3 | 3.67 |
| Matrix | 1, 5, 4 | 10 | 3 | 3.33 |
| Terminator | 4, 2, 5 | 11 | 3 | 3.67 |
| Titanic | 3, 1 | 4 | 2 | 2.00 |

**Prediction:** Ana → Terminator = Terminator's mean = **3.67**.

**What it captures and what it misses.** It captures the general quality of the movie. It gives the *same* prediction to every user, so it is not personalised.

**Where the item mean is used in practice:**

- **The "4.2 ★" on every product page** (Amazon, IMDb, Google Maps) is an item mean, usually a damped one.
- **Fallback for a brand-new user** (we know the item, not the user).
- **A ranking feature**, and the basis of "Top rated" lists (with the correction described below).

### Baseline 4 — the bias baseline $\mu + b_u + b_i$

**Idea.** Combine the last two. Start from the global mean, then add two corrections:

- $b_u$ = how much higher or lower than average *this user* rates,
- $b_i$ = how much higher or lower than average *this movie* is rated.

$$\hat r_{ui} = \mu + b_u + b_i$$

**By hand (simple version).** $b_u = \bar r_u - \mu$ and $b_i = \bar r_i - \mu$:

- $b_{\text{Ana}} = 3.33 - 3.29 = +0.05$ (Ana rates almost exactly like the average user)
- $b_{\text{Terminator}} = 3.67 - 3.29 = +0.38$ (Terminator is rated above average)

$$\hat r = 3.29 + 0.05 + 0.38 = 3.71$$

**Prediction:** Ana → Terminator = **3.71**.

**Why this is better in general.** The same 4-star rating can come from "a generous user and an average movie" or from "a strict user and a great movie". The bias baseline separates the two effects.

**Damping (shrinkage) — the version used in practice.** Titanic has only two ratings. Is it really a 2.0 movie, or were we unlucky with two viewers? With few ratings we should not trust the average fully. So in practice the correction is divided by "count + $\lambda$" instead of "count":

$$b_i = \frac{\sum_{u}(r_{ui}-\mu)}{\lambda + n_i}$$

Example for Titanic with $\lambda = 2$: the deviations from $\mu$ are $(3-3.29) + (1-3.29) = -2.57$.

- without damping: $b_i = -2.57 / 2 = -1.29$
- with damping: $b_i = -2.57 / (2+2) = -0.64$

The correction is cut in half because the evidence is weak. For a movie with 1,000 ratings, adding 2 to the count changes nothing. **That is the whole point: small samples are pulled toward the average, large samples are left alone.**

**Where the bias baseline is used in practice:** it is *the* reference baseline for rating prediction. It was the first building block of the Netflix Prize solutions, it is `BaselineOnly` in the Surprise library, and matrix-factorization models contain exactly these terms: $\hat r_{ui} = \mu + b_u + b_i + \mathbf p_u^\top \mathbf q_i$.

### What the four baselines say about our question

| Baseline | Prediction for Ana → Terminator | Would we recommend it? |
|---|---|---|
| Global mean | 3.29 | maybe |
| User mean | 3.33 | maybe |
| Item mean | 3.67 | yes |
| Bias baseline | 3.71 | yes |

**All four say "Ana will probably like Terminator". All four are wrong.** None of them can know that Ana dislikes action movies, because none of them looks at *which* movies Ana liked. That is exactly the weakness that Part 2 fixes.

In [6]:
mu     = np.nanmean(T.values)
r_user = T.mean(axis=1)
r_item = T.mean(axis=0)
print(f"global mean mu = {mu:.3f}\n")
print("user means:", r_user.round(2).to_dict())
print("item means:", r_item.round(2).to_dict(), "\n")
print(f"Ana -> Terminator   global={mu:.2f}   user mean={r_user['Ana']:.2f}   item mean={r_item['Terminator']:.2f}   "
      f"bias={mu + (r_user['Ana']-mu) + (r_item['Terminator']-mu):.2f}")
dev = (T["Titanic"].dropna() - mu).sum()
print(f"Titanic bias: undamped={dev/2:.2f}   damped (lambda=2)={dev/4:.2f}")

global mean mu = 3.286

user means: {'Ana': 3.33, 'Ben': 3.33, 'Cem': 3.5, 'Dee': 3.0}
item means: {'ToyStory': 3.33, 'Nemo': 3.67, 'Matrix': 3.33, 'Terminator': 3.67, 'Titanic': 2.0} 

Ana -> Terminator   global=3.29   user mean=3.33   item mean=3.67   bias=3.71
Titanic bias: undamped=-1.29   damped (lambda=2)=-0.64


### Measuring a rating predictor: MAE and RMSE

**The situation.** Next week three new ratings arrive. These are the **test set**: ratings the model did not see when it was built.

| User | Movie | True rating |
|---|---|---|
| Ana | Terminator | 1 |
| Ben | Nemo | 2 |
| Dee | Toy Story | 3 |

(So Ana really did hate Terminator.)

**Step 1 — prediction and error for each test rating.** Take the user-mean baseline:

| Test rating | True $r$ | Predicted $\hat r$ | Error $r - \hat r$ | Absolute error | Squared error |
|---|---|---|---|---|---|
| Ana, Terminator | 1 | 3.33 | −2.33 | 2.33 | 5.44 |
| Ben, Nemo | 2 | 3.33 | −1.33 | 1.33 | 1.78 |
| Dee, Toy Story | 3 | 3.00 | 0.00 | 0.00 | 0.00 |

**Step 2 — MAE (Mean Absolute Error):** average of the absolute errors.

$$\text{MAE} = \frac{2.33 + 1.33 + 0.00}{3} = 1.22$$

Meaning: "on average we are 1.22 stars off."

**Step 3 — RMSE (Root Mean Squared Error):** average the squared errors, then take the square root.

$$\text{RMSE} = \sqrt{\frac{5.44 + 1.78 + 0.00}{3}} = \sqrt{2.41} = 1.55$$

**Why two metrics?** Squaring makes big errors count much more. The one large error (2.33) contributes 5.44 of the total 7.22 in RMSE, which is 75%, but only 64% of the MAE total. So:

- use **MAE** when every star of error costs the same; it is easy to explain;
- use **RMSE** when large mistakes are especially bad (recommending something the user hates is worse than being slightly off twice).
- **RMSE ≥ MAE always.** They are equal only when all errors have the same size.

**The general formulas** ($T$ = the test set):

$$\text{MAE} = \frac{1}{|T|}\sum_{(u,i)\in T} |r_{ui}-\hat r_{ui}| \qquad \text{RMSE} = \sqrt{\frac{1}{|T|}\sum_{(u,i)\in T} (r_{ui}-\hat r_{ui})^2}$$

**Why must the averages be computed from the training data only?** If the test ratings were included when computing $\mu$ or $\bar r_u$, the model would already "know" part of the answer, and the measured error would look better than it really is. This is called **data leakage**. In real work it is the most common reason why a model looks great offline and fails in production.

In [7]:
test_tiny = [("Ana", "Terminator", 1), ("Ben", "Nemo", 2), ("Dee", "ToyStory", 3)]
y_true = np.array([r for _, _, r in test_tiny])
for name, f in [("global mean", lambda u, i: mu), ("user mean", lambda u, i: r_user[u]), ("item mean", lambda u, i: r_item[i])]:
    p = np.array([f(u, i) for u, i, _ in test_tiny])
    print(f"{name:12s} predictions={p.round(2)}  MAE={np.abs(y_true-p).mean():.2f}  RMSE={np.sqrt(((y_true-p)**2).mean()):.2f}")

global mean  predictions=[3.29 3.29 3.29]  MAE=1.29  RMSE=1.52
user mean    predictions=[3.33 3.33 3.  ]  MAE=1.22  RMSE=1.55
item mean    predictions=[3.67 3.67 3.33]  MAE=1.56  RMSE=1.83


With three test ratings the ranking of the baselines is pure luck. On real data (section 1.D) the order is clear and stable.

### Top-N baselines: what do we put on the home page?

Now the other task. We do not predict a rating; we must choose which movies to **show**. Every item gets a **score**, and we show the items with the highest scores.

Take four movies from a bigger catalog, described only by their statistics:

| Movie | Number of ratings | Average rating |
|---|---|---|
| Blockbuster | 200 | 4.2 |
| Classic | 150 | 4.5 |
| Niche gem | 1 | 5.0 |
| Flop | 80 | 2.1 |

**Baseline A — Popularity.** Score = number of ratings (how many people interacted).

Ranking: Blockbuster (200), Classic (150), Flop (80), Niche gem (1).

- *Why it works:* most people like what most people like, and a popular item is a safe guess for a user we know nothing about.
- *Weakness:* it counts interactions, not satisfaction. "Flop" is third even though people hate it. And every user gets the same list.
- *Where used:* "Trending", "Most popular", "Best sellers"; the default for new users; a required comparison in every experiment.

**Baseline B — Highest average rating.** Score = average rating.

Ranking: Niche gem (5.0), Classic (4.5), Blockbuster (4.2), Flop (2.1).

- *Weakness:* "Niche gem" is first because **one** person gave it 5 stars. An average of one rating is not reliable. On real data the top of this list is full of unknown items with one or two perfect ratings, which is why the professor finds this baseline weak.

**Baseline C — Damped (Bayesian) average.** This is the fix, and it is what real "Top rated" lists use. Pretend every movie starts with $\lambda$ imaginary ratings equal to the global mean $\mu$:

$$\text{score}_i = \frac{\text{(sum of ratings)} + \lambda\,\mu}{n_i + \lambda}$$

By hand with $\lambda = 10$ and $\mu = 3.5$ (sum of ratings = count × average):

| Movie | Sum of ratings | Calculation | Score |
|---|---|---|---|
| Blockbuster | $200 \times 4.2 = 840$ | $(840 + 35)/210$ | 4.17 |
| Classic | $150 \times 4.5 = 675$ | $(675 + 35)/160$ | 4.44 |
| Niche gem | $1 \times 5.0 = 5$ | $(5 + 35)/11$ | 3.64 |
| Flop | $80 \times 2.1 = 168$ | $(168 + 35)/90$ | 2.26 |

Ranking: Classic, Blockbuster, Niche gem, Flop. The movies with many ratings barely move (4.5 → 4.44). The movie with one rating drops from 5.0 to 3.64, close to the global mean. This is the same shrinkage idea as in the bias baseline, and it is how the IMDb Top 250 is calculated.

### Measuring a Top-N recommender: Hit Ratio

We cannot use MAE here because there are no predicted ratings. Instead we ask: **does a movie the user really liked appear in our list?**

**The protocol the professor uses ("1 positive + 99 negatives").** For each user:

1. take one movie the user rated 5 stars — the **positive**;
2. take 99 random movies the user has never rated — the **negatives** (we assume the user is not interested in them);
3. let the recommender score all 100 and sort them;
4. if the positive is among the first $K$, count a **hit**.

$$\text{HR@}K = \frac{\text{number of hits}}{\text{number of users tested}}$$

**By hand, with 1 positive + 4 negatives to keep it small.** The recommender is Popularity, $K = 2$. The numbers are rating counts.

| User | Positive (count) | Counts of the 4 negatives | Sorted counts | Rank of the positive | Hit@2? |
|---|---|---|---|---|---|
| U1 | Blockbuster (200) | 3, 150, 1, 12 | **200**, 150, 12, 3, 1 | 1 | hit |
| U2 | Niche gem (1) | 80, 2, 0, 5 | 80, 5, 2, **1**, 0 | 4 | miss |
| U3 | Classic (150) | 200, 7, 1, 80 | 200, **150**, 80, 7, 1 | 2 | hit |

$$\text{HR@2} = \frac{2}{3} = 0.67$$

**What would a random recommender get?** It puts the positive in each of the 5 positions with the same probability, so the chance of being in the top 2 is $2/5 = 0.40$. With 100 candidates:

$$\text{HR@}K \text{ of a random recommender} = \frac{K}{100} \quad\Rightarrow\quad \text{HR@5}=0.05,\ \text{HR@10}=0.10,\ \text{HR@20}=0.20$$

Memorise this. It is the "zero level" for this protocol, and a likely exam question.

**Why 99 sampled negatives and not the whole catalog?** Scoring and sorting every item for every user is expensive, so researchers sample. It is cheaper, but it changes the meaning of the number. Section 1.D shows how much.

**Look at user U2.** Popularity fails for the person with niche taste. That is the cost of having no personalisation, and you cannot see it in the average HR.

### The algorithms in general form

```
FIT_BASELINES(train):                       # one pass over the data: O(|R|)
    mu          <- mean of all ratings
    user_avg[u] <- mean of the ratings of user u
    item_avg[i] <- mean of the ratings of item i
    pop[i]      <- number of ratings of item i

PREDICT_ITEM_AVG(u, i):                     # O(1)
    if i in item_avg: return item_avg[i]
    else:             return mu             # fallback for an unseen item

EVALUATE_HIT_RATIO(score, test_users, K):
    hits <- 0
    for each user u:
        pos  <- one held-out item u liked            # must NOT be in the training data
        negs <- 99 random items u has NOT interacted with
        rank the 100 candidates by score(item), highest first
        if pos is within the first K: hits <- hits + 1
    return hits / number of users
```

**Cost.** Building the tables is one pass over the ratings, $O(|R|)$. One prediction is a table lookup, $O(1)$. This is why baselines are essentially free, and why they are always available as a fallback.

## 1.C The professor's code

`baselines.ipynb` has three blocks.

**Block 1 — random prediction.**

```python
error = 0
for i in range(n_ratings):
    error += np.abs(ratings.iloc[i,2] - np.random.randint(1,6))
print("MAE:", error/n_ratings)
```

For every rating in the dataset it draws a random integer from 1 to 5 and accumulates the absolute error. No train/test split is needed because nothing is learned. This is the "worst sensible" reference (MAE ≈ 1.48).

**Block 2 — average predictions.**

```python
X_train, X_test = train_test_split(ratings, test_size=0.1)       # 90% train, 10% test
avg_rating    = X_train.iloc[:,2].mean()                         # global mean, from TRAIN only
item_averages = X_train.groupby('movieId')['rating'].mean()      # table: movieId -> mean
user_averages = X_train.groupby('userId')['rating'].mean()       # table: userId  -> mean

for index, row in X_test.iterrows():
    pred_rating = item_averages.get(movie_id, avg_rating)        # item mean, or global mean if the item is unseen
    error_item_avg += np.abs(true_rating - pred_rating)
```

`groupby('movieId')['rating'].mean()` does in one line what we did by hand in the item-mean table: it groups the rows by movie and averages each group. `.get(movie_id, avg_rating)` is the fallback: some test movies never appear in the training set, and for those there is no item mean.

**Block 3 — Top-N with Hit Ratio.** The same loop is written three times, once per scoring rule (popularity, random, average rating). It is the `EVALUATE_HIT_RATIO` pseudocode above.

A short version of block 2, in his style, so you can run it:

In [8]:
X_train, X_test = train_test_split(ratings, test_size=0.1, random_state=SEED)

avg_rating    = X_train.rating.mean()                       # global mean (only from train!)
item_averages = X_train.groupby("movieId")["rating"].mean() # Series: movieId -> mean
user_averages = X_train.groupby("userId")["rating"].mean()

err = 0
for _, row in X_test.iterrows():                            # professor-style explicit loop
    pred = item_averages.get(row["movieId"], avg_rating)    # fallback for unseen items
    err += abs(row["rating"] - pred)
print("Item-average MAE (loop version):", round(err/len(X_test), 4))

Item-average MAE (loop version): 0.761


### Why this code is fine for a class but not how it is done at work

| What the professor does | Why it is a problem in practice | What is done instead |
|---|---|---|
| Python `for` loop over rows with `.iloc[i]` / `.iterrows()` | Each step creates a pandas object; orders of magnitude slower than a vectorised operation. Fine for 100k rows, impossible for 100M. | `test.movieId.map(item_averages).fillna(mu)` — one vectorised line (section 1.D) |
| The same 40-line evaluation loop copied three times | Every bug fix must be made three times; the copies drift apart | One evaluation function that takes the scoring rule as an argument |
| Random prediction evaluated on all data, the others on the test split | The numbers are not measured on the same ratings | One split, one evaluation function for all methods |
| Random draws integers 1..5, but real ratings include half stars (0.5 … 5) | The baseline does not match the data | Sample from the observed rating distribution |
| The positive item is taken from the full `ratings`, while popularity is computed on `X_train` (90% of the data) | **Data leakage:** the rating being tested is usually inside the training data | Hold the positive out of the training data (leave-one-out) |
| Plain average for "highest average rating" | Items with one rating dominate | Damped average |

**One more point that is easy to miss.** The professor reports HR@20 > 0.91 for popularity and calls it a very strong baseline. A large part of that number comes from the *protocol*, not from popularity itself: the positive is a movie the user really watched (so it tends to be popular), while the 99 negatives are drawn uniformly from the catalog, and half of the catalog has only 1–2 ratings. Beating unknown movies with a popular one is easy. Section 1.D tests this claim.

## 1.D The industry version

### Rating prediction with vectorised pandas

The same four baselines, without any Python loop. `map` looks up each test row in the table of averages; `fillna(mu)` is the fallback.

In [9]:
def mae(y, p):  return float(np.mean(np.abs(y - p)))
def rmse(y, p): return float(np.sqrt(np.mean((y - p) ** 2)))

def fit_bias_baseline(train, lam_i=10, lam_u=5):
    """r_hat = mu + b_u + b_i  with damped (shrunk) biases."""
    mu  = train.rating.mean()
    d   = train.assign(res=train.rating - mu)
    g   = d.groupby("movieId").res
    b_i = g.sum() / (lam_i + g.count())
    d["res2"] = d.res - d.movieId.map(b_i)                 # remove item effect before fitting user effect
    g   = d.groupby("userId").res2
    b_u = g.sum() / (lam_u + g.count())
    return mu, b_u, b_i

y  = X_test.rating.to_numpy()
mu = avg_rating
mu_, b_u, b_i = fit_bias_baseline(X_train)

preds = {
    "Random (1..5)":      np.random.default_rng(SEED).integers(1, 6, len(X_test)),
    "Global mean":        np.full(len(X_test), mu),
    "Item mean":          X_test.movieId.map(item_averages).fillna(mu).to_numpy(),
    "User mean":          X_test.userId.map(user_averages).fillna(mu).to_numpy(),
    "Bias: mu+b_u+b_i":   np.clip(mu_ + X_test.userId.map(b_u).fillna(0) + X_test.movieId.map(b_i).fillna(0), 0.5, 5).to_numpy(),
}
rating_table = pd.DataFrame({k: {"MAE": mae(y, p), "RMSE": rmse(y, p)} for k, p in preds.items()}).T
rating_table

,MAE,RMSE
Random (1..5),1.4787,1.8267
Global mean,0.8351,1.0506
Item mean,0.7610,0.9825
User mean,0.7432,0.9563
Bias: mu+b_u+b_i,0.6796,0.8861


**Reading the table.** Each row adds a little more information: nothing → the global mean → the user or the item → both. The error drops at every step. The bias baseline is clearly the best, with three lines of code. **This is the number a collaborative-filtering model has to beat**, not the global mean.

### The same baselines with the Surprise library

`scikit-surprise` is the standard Python library for **rating prediction**. It is used for teaching, research and quick prototypes (not for large production systems). `NormalPredictor` is a random baseline and `BaselineOnly` is $\mu + b_u + b_i$ with damping. Cross-validation is built in, so you get a more reliable estimate than from one split.

In [10]:
from surprise import Dataset, Reader, NormalPredictor, BaselineOnly
from surprise.model_selection import cross_validate

sdata = Dataset.load_from_df(ratings[["userId", "movieId", "rating"]], Reader(rating_scale=(0.5, 5)))
for name, algo in [("NormalPredictor (random)", NormalPredictor()), ("BaselineOnly (mu+b_u+b_i)", BaselineOnly(verbose=False))]:
    cv = cross_validate(algo, sdata, measures=["MAE", "RMSE"], cv=3, verbose=False)
    print(f"{name:28s} MAE={cv['test_mae'].mean():.4f}  RMSE={cv['test_rmse'].mean():.4f}")

NormalPredictor (random)     MAE=1.1410  RMSE=1.4304


BaselineOnly (mu+b_u+b_i)    MAE=0.6769  RMSE=0.8773


(`NormalPredictor` draws from a normal distribution fitted to the ratings instead of a uniform one, which is why its error is lower than our uniform random baseline.)

### Top-N baselines: the professor's protocol, done cleanly

Three changes compared with his code: the positive is **removed from the training data** (no leakage), there is **one** evaluation function, and ties are broken randomly.

In [11]:
def leave_one_out(ratings, pos_thr=5.0, seed=SEED):
    """Hold out ONE positive (rating >= pos_thr) per user as test; everything else is train."""
    test  = ratings[ratings.rating >= pos_thr].groupby("userId").sample(n=1, random_state=seed)
    train = ratings.drop(test.index)
    return train, test

all_items  = ratings.movieId.unique()
seen_items = ratings.groupby("userId").movieId.apply(set).to_dict()

def sampled_hit_ratio(score_fn, test, Ks=(5, 10, 20), n_neg=99, neg_p=None, seed=SEED):
    """score_fn(array of movieIds) -> array of scores.  Protocol: 1 positive + n_neg sampled negatives."""
    rng, hits = np.random.default_rng(seed), dict.fromkeys(Ks, 0)
    for u, pos in zip(test.userId, test.movieId):
        negs = []
        while len(negs) < n_neg:                                   # sample items the user has not rated
            negs += [m for m in rng.choice(all_items, 2 * n_neg, p=neg_p) if m not in seen_items[u]]
        cand = np.array([pos] + negs[:n_neg])
        s    = score_fn(cand) + rng.random(len(cand)) * 1e-9       # random tie-breaking
        rank = int((s > s[0]).sum())                               # how many candidates beat the positive
        for k in Ks: hits[k] += rank < k
    return {f"HR@{k}": hits[k] / len(test) for k in Ks}

tr, te = leave_one_out(ratings)
mu_tr  = tr.rating.mean()
g      = tr.groupby("movieId").rating
pop, avg = g.count(), g.mean()
LAM = 20
damped = (g.sum() + LAM * mu_tr) / (g.count() + LAM)               # Bayesian / damped average

scorers = {
    "Random":           lambda c: np.zeros(len(c)),
    "Highest average":  lambda c: avg.reindex(c).fillna(mu_tr).to_numpy(),
    "Damped average":   lambda c: damped.reindex(c).fillna(mu_tr).to_numpy(),
    "Popularity":       lambda c: pop.reindex(c).fillna(0).to_numpy(),
}
topn_table = pd.DataFrame({k: sampled_hit_ratio(f, te) for k, f in scorers.items()}).T
print("users evaluated:", len(te))
topn_table

users evaluated: 573


,HR@5,HR@10,HR@20
Random,0.0628,0.1134,0.2269
Highest average,0.0209,0.1449,0.3997
Damped average,0.5672,0.6579,0.7347
Popularity,0.6754,0.7853,0.8761


**Reading the table.**

- **Random** is close to the theoretical $K/100$.
- **Highest average** is very weak; at HR@5 it is even below random, because the top of the list is full of movies with one perfect rating.
- **Damped average** fixes exactly that, with one extra term in the formula.
- **Popularity** is the strongest.

### Testing the claim: how much of popularity's score comes from the protocol?

Same recommender, same users. The only change: the 99 negatives are sampled **in proportion to their popularity** (so they are realistic competitors) instead of uniformly.

In [12]:
p_pop = pop.reindex(all_items).fillna(0).to_numpy(); p_pop = p_pop / p_pop.sum()
pd.DataFrame({
    "Popularity | uniform negatives":            sampled_hit_ratio(scorers["Popularity"], te),
    "Popularity | popularity-sampled negatives": sampled_hit_ratio(scorers["Popularity"], te, neg_p=p_pop),
}).T

,HR@5,HR@10,HR@20
Popularity | uniform negatives,0.6754,0.7853,0.8761
Popularity | popularity-sampled negatives,0.2182,0.3386,0.4747


The score falls to less than half. Nothing about the recommender changed; only the difficulty of the negatives did.

**What this means in real work:**

- An offline number is only meaningful together with its protocol. Never compare Hit Ratios measured in different ways.
- Teams therefore evaluate on the **full catalog** (as we do in Parts 2 and 3) or with popularity-aware negatives.
- Accuracy is never reported alone. **Coverage** (what share of the catalog is ever recommended), **novelty** and **diversity** are reported next to it. Popularity recommends the same handful of items to everyone, so its coverage is close to zero.
- The final decision is made with an **online A/B test** (does watch time or revenue go up?), because offline metrics only approximate what users will do.

## 1.E Exam — Baselines

> The "professor's" questions are the ones that appear in the text of his notebook; the rest are my predictions of other forms the same material could take.

**Q (from the notebook's conclusion). Why is popularity such a strong Top-N baseline, and why is "highest average rating" weak?**

*Popularity:* Interaction data follows a long-tail distribution — a small set of items receives most interactions, so a randomly held-out positive item is very likely to be a popular one. Under the 1-positive + 99-random-negatives protocol the negatives are mostly unpopular long-tail items, which makes ranking by count very effective.
*Highest average:* The mean of very few ratings is unreliable. An item with a single 5-star rating gets the maximum score and outranks broadly liked items with hundreds of ratings. The fix is to account for support: require a minimum number of ratings or use a damped (Bayesian) average $\frac{\sum r + \lambda\mu}{n+\lambda}$.

**Q. Why do we need baselines at all?**
They give a minimum performance threshold. A complex model is only justified if it clearly beats cheap heuristics; baselines also serve as cold-start and fallback recommenders in production.

**Q. What HR@10 do you expect from a random recommender under the 1+99 protocol? Why?**
$10/100 = 0.10$. The positive item is equally likely to land in any of the 100 positions, so the probability of being in the top $K$ is $K/100$.

**Q. MAE vs RMSE — which one is larger and what is the difference?**
RMSE ≥ MAE always. RMSE squares the errors, so it penalizes large errors more heavily; MAE weights all errors linearly and is more robust to outliers.

**Q (calculation). True ratings $[4, 2, 5, 3]$, predictions $[3.5, 3.5, 3.5, 3.5]$. Compute MAE and RMSE.**
Errors: $0.5, 1.5, 1.5, 0.5$ → MAE $= 4/4 = 1.0$. Squared: $0.25, 2.25, 2.25, 0.25$ → mean $=1.25$ → RMSE $=\sqrt{1.25}\approx 1.118$.

**Q. Why are the averages computed on the training set only?**
Computing them on all data leaks test ratings into the model (data leakage) and gives an optimistically biased error estimate.

**Q. What happens for a user/item in the test set that never appears in training?**
There is no user/item average — this is the cold-start case. The code falls back to the global average (`.get(id, avg_rating)`).

**Q. What is the main weakness of the popularity recommender even though its hit ratio is high?**
It is not personalized: every user receives the same list. It has near-zero catalog coverage and novelty, and it reinforces popularity bias (rich-get-richer feedback loop).

**Q (algorithm). Write the pseudocode for evaluating a recommender with Hit Ratio@K.** → write the pseudocode from section 1.B. Three points that must not be missing: the negatives are chosen from items the user has **not rated**, the positive must have been **removed** from train, and the denominator is the **number of test cases**.

---
# Part 2 — Neighborhood-based Collaborative Filtering (kNN)

## 2.A The real-world problem

**Situation.** StreamBox now has users and ratings. The product manager wants two things:

1. A row on the home page: **"Because you watched Toy Story"**.
2. Fewer bad recommendations: *"Ana gave Matrix one star and we still showed her Terminator."*

The baselines of Part 1 cannot do either. They gave Ana → Terminator a score of 3.3–3.7 because they never look at *which* movies Ana liked.

**Where this exact problem appears in real products:**

- **Amazon:** "Customers who bought this item also bought…". This is the original use; Amazon published the method in 2003 and the professor's notebook cites that paper.
- **Netflix:** "Because you watched …" rows.
- **Spotify / YouTube:** "Fans also like", "Up next".
- **Any online shop:** "Similar products", "Frequently bought together".

**The idea: collaborative filtering (CF).** Use only the rating table itself. No genres, no descriptions, no actors. The assumption is:

> People who agreed in the past will agree in the future.

We never told the system that Matrix and Terminator are both action movies. It will discover that they belong together only because *the same people like both*. That is what "collaborative" means, and it is why CF works for any kind of item (movies, songs, shoes) without understanding the items.

**Two versions:**

| | User-based CF | Item-based CF (the professor's version) |
|---|---|---|
| Question | "Which users are similar to Ana? What did they like?" | "Which movies are similar to the ones Ana liked?" |
| Similarity is computed between | two rows of the table (users) | two columns of the table (items) |

**Why industry chose item-based:**

1. **Stability.** A user's row changes with every click. The relation "Matrix is similar to Terminator" changes slowly. So item similarities can be computed once, at night, and reused all day.
2. **Explainability.** "Because you watched Toy Story" is a reason the user understands and trusts.
3. **Size.** Many services have far more users than items, so the item–item table is the smaller one.

**When kNN-CF is the right tool:** you have interaction data, you want a strong method quickly, and you need explainable results. It needs no training.
**When it fails:** a brand-new item or user has no interactions (cold start); very sparse data gives unreliable similarities; and comparing all pairs of items costs $O(n^2)$.

## 2.B Solving it by hand

Our table again:

| | Toy Story | Nemo | Matrix | Terminator | Titanic |
|---|---|---|---|---|---|
| **Ana** | 5 | 4 | 1 | ? | ? |
| **Ben** | 1 | ? | 5 | 4 | ? |
| **Cem** | 4 | 5 | ? | 2 | 3 |
| **Dee** | ? | 2 | 4 | 5 | 1 |

**Goal:** predict Ana → Terminator. The plan has two steps:

1. measure how similar Terminator is to each movie Ana has already rated (Toy Story, Nemo, Matrix);
2. average Ana's ratings of those movies, giving more weight to the movies that are more similar to Terminator.

The logic: *if Terminator is very similar to Matrix, then Ana's opinion of Matrix tells us her opinion of Terminator.*

Everything depends on step 1, the similarity. We compute it in three ways and see that they give three different answers.

### Step 1a — Jaccard similarity

**Idea.** Ignore the rating values. Only ask: *who rated this movie?* Two movies are similar if the same people rated them.

$$J(i,j) = \frac{|U_i \cap U_j|}{|U_i \cup U_j|} = \frac{\text{users who rated both}}{\text{users who rated at least one}}$$

**By hand.** First write the set of users for each movie (read each column):

| Movie | Users who rated it |
|---|---|
| Toy Story | {Ana, Ben, Cem} |
| Nemo | {Ana, Cem, Dee} |
| Matrix | {Ana, Ben, Dee} |
| Terminator | {Ben, Cem, Dee} |
| Titanic | {Cem, Dee} |

Terminator vs Toy Story:

- both: {Ben, Cem} → 2 users
- at least one: {Ana, Ben, Cem, Dee} → 4 users
- $J = 2/4 = 0.50$

Terminator vs Nemo: both = {Cem, Dee} = 2; at least one = 4 → $0.50$
Terminator vs Matrix: both = {Ben, Dee} = 2; at least one = 4 → $0.50$

**Result:** Jaccard says Terminator is *equally* similar to Toy Story, Nemo and Matrix. It cannot tell them apart, because it does not look at whether people liked or disliked them.

**When Jaccard is the right choice.** For **implicit data**, where there are no rating values at all: purchases, clicks, plays. "People who bought X also bought Y" is a co-occurrence count, and Jaccard is exactly a normalised co-occurrence count. On large sparse data it works well, because the fact that the same people chose two items is already a strong signal.

### Step 1b — Cosine similarity

**Idea.** Use the rating values. Treat each movie as a vector of ratings and measure the angle between two vectors. Same direction → 1. Unrelated → 0.

$$\cos(i,j) = \frac{\sum_u r_{ui}\, r_{uj}}{\sqrt{\sum_u r_{ui}^2}\ \sqrt{\sum_u r_{uj}^2}}$$

- The **numerator** (dot product) adds up, user by user, "rating of $i$ × rating of $j$". It is large when the same users give high ratings to both.
- The **denominator** divides by the lengths of the two vectors, so that a movie does not look similar to everything just because its numbers are big.

**Which users go into the sums?** The professor uses only the users who rated **both** movies (the "co-rated" users). We do the same.

**By hand — Terminator vs Toy Story.** Users who rated both: Ben and Cem.

| | Ben | Cem |
|---|---|---|
| Terminator | 4 | 2 |
| Toy Story | 1 | 4 |

- dot product: $4 \times 1 + 2 \times 4 = 4 + 8 = 12$
- length of Terminator: $\sqrt{4^2 + 2^2} = \sqrt{20} = 4.47$
- length of Toy Story: $\sqrt{1^2 + 4^2} = \sqrt{17} = 4.12$
- $\cos = 12 / (4.47 \times 4.12) = 12 / 18.44 = 0.65$

**Terminator vs Nemo.** Co-rated users: Cem and Dee. Terminator = [2, 5], Nemo = [5, 2].

- dot: $2\times5 + 5\times2 = 20$; lengths: $\sqrt{29}$ and $\sqrt{29}$ → $\cos = 20/29 = 0.69$

**Terminator vs Matrix.** Co-rated users: Ben and Dee. Terminator = [4, 5], Matrix = [5, 4].

- dot: $4\times5 + 5\times4 = 40$; lengths: $\sqrt{41}$ and $\sqrt{41}$ → $\cos = 40/41 = 0.98$

**Look closely at Terminator vs Nemo.** Cem gave Terminator 2 and Nemo 5. Dee gave Terminator 5 and Nemo 2. They are *exact opposites*: whoever likes one dislikes the other. And cosine still says 0.69, "quite similar".

**Why does this happen?** All ratings are positive numbers (1 to 5). So every product $r_{ui} \times r_{uj}$ is positive, and every user *adds* to the similarity, even a user who disagrees. A disagreement (5 and 1 → product 5) even adds more than an agreement that both movies are bad (1 and 1 → product 1). Raw cosine on star ratings has no way to say "these two are opposites".

### Step 1c — Adjusted cosine (mean-centered cosine)

**Idea.** Before comparing, turn each rating into "how much above or below this user's own average". Then:

- a **positive** number means "this user liked it more than they usually like things",
- a **negative** number means "this user liked it less than usual".

Now, when two users disagree, one number is positive and the other negative, the product is **negative**, and it *reduces* the similarity. Agreement and disagreement finally have opposite effects.

**By hand — first build the centered table.** Subtract each user's mean (from Part 1: Ana 3.33, Ben 3.33, Cem 3.50, Dee 3.00) from each of their ratings:

| | Toy Story | Nemo | Matrix | Terminator | Titanic |
|---|---|---|---|---|---|
| **Ana** (−3.33) | +1.67 | +0.67 | −2.33 | ? | ? |
| **Ben** (−3.33) | −2.33 | ? | +1.67 | +0.67 | ? |
| **Cem** (−3.50) | +0.50 | +1.50 | ? | −1.50 | −0.50 |
| **Dee** (−3.00) | ? | −1.00 | +1.00 | +2.00 | −2.00 |

You can already read taste from the signs: Ana is + on the animation movies and − on Matrix. Ben is the reverse.

**Terminator vs Toy Story** (Ben, Cem): Terminator = [+0.67, −1.50], Toy Story = [−2.33, +0.50]

- dot: $(0.67)(-2.33) + (-1.50)(0.50) = -1.56 - 0.75 = -2.31$
- lengths: $\sqrt{0.67^2 + 1.50^2} = 1.64$ and $\sqrt{2.33^2 + 0.50^2} = 2.39$
- similarity $= -2.31 / (1.64 \times 2.39) = -0.59$

**Terminator vs Nemo** (Cem, Dee): Terminator = [−1.50, +2.00], Nemo = [+1.50, −1.00]

- dot: $(-1.50)(1.50) + (2.00)(-1.00) = -2.25 - 2.00 = -4.25$
- lengths: $\sqrt{1.50^2 + 2.00^2} = 2.50$ and $\sqrt{1.50^2 + 1.00^2} = 1.80$
- similarity $= -4.25 / (2.50 \times 1.80) = -0.94$

**Terminator vs Matrix** (Ben, Dee): Terminator = [+0.67, +2.00], Matrix = [+1.67, +1.00]

- dot: $(0.67)(1.67) + (2.00)(1.00) = 1.11 + 2.00 = 3.11$
- lengths: $\sqrt{0.67^2 + 2.00^2} = 2.11$ and $\sqrt{1.67^2 + 1.00^2} = 1.94$
- similarity $= 3.11 / (2.11 \times 1.94) = +0.76$

### The three similarities side by side

| Terminator compared with | Jaccard | Cosine (raw) | Adjusted cosine | What a human would say |
|---|---|---|---|---|
| Toy Story | 0.50 | 0.65 | **−0.59** | different |
| Nemo | 0.50 | 0.69 | **−0.94** | opposite |
| Matrix | 0.50 | 0.98 | **+0.76** | similar |

**Only adjusted cosine matches common sense:** Terminator is like Matrix and unlike the animation movies.

- Jaccard gives the same value to all three because it ignores the rating values.
- Raw cosine gives all three a fairly high positive value because it cannot express disagreement.
- Adjusted cosine has both signs, and puts the movies on the correct sides.

Ranges to remember: Jaccard is in [0, 1]. Cosine on positive ratings is in [0, 1]. Adjusted cosine (and Pearson correlation, which is the same idea with item means) is in [−1, +1].

### Step 2 — from similarities to a prediction

**The formula (weighted average).** Use Ana's own ratings; weight each one by how similar that movie is to Terminator:

$$\hat r_{ui} = \frac{\sum_{j} \text{sim}(i,j)\cdot r_{uj}}{\sum_{j} |\text{sim}(i,j)|} \qquad j \text{ runs over the movies user } u \text{ has rated}$$

**Why divide by the sum of the similarities?** So the result is a proper average and stays on the 1–5 scale. If all similarities are equal, the formula gives the plain average of the user's ratings.

**Prediction with Jaccard.** All three weights are 0.50:

$$\hat r = \frac{0.50\times5 + 0.50\times4 + 0.50\times1}{0.50+0.50+0.50} = \frac{5.00}{1.50} = 3.33$$

This is exactly Ana's mean. With equal weights the method adds nothing to the user-mean baseline.

**Prediction with raw cosine.** Ana's ratings: Toy Story 5, Nemo 4, Matrix 1.

$$\hat r = \frac{0.65\times5 + 0.69\times4 + 0.98\times1}{0.65+0.69+0.98} = \frac{3.25 + 2.76 + 0.98}{2.32} = \frac{6.99}{2.32} = 3.02$$

A little lower than the baselines (because Matrix has the largest weight), but still "she will find it okay".

**Prediction with adjusted cosine.** Because the similarities are computed on centered ratings, the prediction is also built from centered ratings: start from Ana's mean and add the weighted average of her deviations.

$$\hat r_{ui} = \bar r_u + \frac{\sum_j \text{sim}(i,j)\,(r_{uj}-\bar r_u)}{\sum_j |\text{sim}(i,j)|}$$

| Movie $j$ | sim(Terminator, $j$) | Ana's centered rating | Product | In words |
|---|---|---|---|---|
| Toy Story | −0.59 | +1.67 | −0.98 | she liked a movie that is the *opposite* of Terminator → push down |
| Nemo | −0.94 | +0.67 | −0.63 | same → push down |
| Matrix | +0.76 | −2.33 | −1.77 | she disliked a movie that is *similar* to Terminator → push down |

- sum of products: $-0.98 - 0.63 - 1.77 = -3.38$
- sum of absolute similarities: $0.59 + 0.94 + 0.76 = 2.29$

$$\hat r = 3.33 + \frac{-3.38}{2.29} = 3.33 - 1.48 = 1.86$$

### The answer to our running question

| Method | Prediction for Ana → Terminator | Recommend? |
|---|---|---|
| Baselines (Part 1) | 3.29 – 3.71 | yes — wrong |
| Item-kNN, Jaccard | 3.33 | maybe — wrong |
| Item-kNN, raw cosine | 3.02 | maybe — wrong |
| **Item-kNN, adjusted cosine** | **1.86** | **no — correct** |
| What Ana really rated | 1 | |

Three evidence lines all point the same way, and the method combined them. This is the first method in the course that gets our question right, and the reason is the mean-centering.

**The parameter $k$.** In a real system Ana may have rated 500 movies. We do not use all of them, only the $k$ movies most similar to the target (the "$k$ nearest neighbours").

- small $k$ (e.g. 3): the prediction depends on very few movies, so it is noisy;
- very large $k$: many weakly related movies enter, their weights add up, and the prediction drifts toward the user's plain average;
- typical values are 20–100, chosen by testing on held-out data.

(The professor's code uses *all* rated items, no $k$. His exercise Q4 uses $k=2$.)

In [13]:
def jaccard(M, i, j):
    a, b = M[i].notna(), M[j].notna()
    return (a & b).sum() / (a | b).sum()

def cosine_corated(M, i, j, center=False):
    X = M.sub(M.mean(axis=1), axis=0) if center else M        # center=True: subtract each USER's mean
    both = M[i].notna() & M[j].notna()                        # users who rated both items
    a, b = X.loc[both, i], X.loc[both, j]
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

rated = ["ToyStory", "Nemo", "Matrix"]                        # what Ana has rated
sims_t = pd.DataFrame({"Jaccard":         [jaccard(T, "Terminator", j) for j in rated],
                       "Cosine":          [cosine_corated(T, "Terminator", j) for j in rated],
                       "Adjusted cosine": [cosine_corated(T, "Terminator", j, center=True) for j in rated]}, index=rated)
print("Similarity of Terminator to each movie Ana rated:"); print(sims_t.round(2), "\n")

r_ana, mean_ana = T.loc["Ana", rated], T.loc["Ana"].mean()
for m in ["Jaccard", "Cosine"]:
    print(f"prediction with {m:8s}: {(sims_t[m] * r_ana).sum() / sims_t[m].abs().sum():.2f}")
s = sims_t["Adjusted cosine"]
print(f"prediction with adjusted cosine: {mean_ana + (s * (r_ana - mean_ana)).sum() / s.abs().sum():.2f}")

Similarity of Terminator to each movie Ana rated:
          Jaccard  Cosine  Adjusted cosine
ToyStory      0.5    0.65            -0.59
Nemo          0.5    0.69            -0.94
Matrix        0.5    0.98             0.76 

prediction with Jaccard : 3.33
prediction with Cosine  : 3.02
prediction with adjusted cosine: 1.86


### Step 3 — Top-N: building the "Because you watched…" row

Predicting one rating is not the product. The product is a **list**. There are two ways to produce it.

**Way 1 — Predict & Sort.** Predict a rating for every movie the user has not seen, sort, take the top N.

*Why it is not used:* for one user and a catalog of 1,000,000 movies this means 1,000,000 predictions, each needing hundreds of similarity computations, and all of it must finish in a few milliseconds while the page loads. And the same similarities are recomputed for every user.

**Way 2 — the kNN-map (what Amazon does).** Split the work into two phases:

- **Offline (at night):** for every item, compute and store its $k$ most similar items. This table is the **kNN-map**.
- **Online (when the page loads):** take the items the user already has, look up their neighbours in the table, add up the scores, and return the best ones. No similarity is computed at request time.

**By hand.** We use Jaccard here, as the professor does. With 5 movies there are $5 \times 4 / 2 = 10$ pairs. The user sets are in Step 1a.

| Pair | Rated both | Rated at least one | Jaccard |
|---|---|---|---|
| Toy Story – Nemo | Ana, Cem | 4 | 0.50 |
| Toy Story – Matrix | Ana, Ben | 4 | 0.50 |
| Toy Story – Terminator | Ben, Cem | 4 | 0.50 |
| Toy Story – Titanic | Cem | 4 | 0.25 |
| Nemo – Matrix | Ana, Dee | 4 | 0.50 |
| Nemo – Terminator | Cem, Dee | 4 | 0.50 |
| Nemo – Titanic | Cem, Dee | 3 | 0.67 |
| Matrix – Terminator | Ben, Dee | 4 | 0.50 |
| Matrix – Titanic | Dee | 4 | 0.25 |
| Terminator – Titanic | Cem, Dee | 3 | 0.67 |

That table *is* the kNN-map (with only 5 movies we keep all neighbours; in real life we would keep the best 30 or so per movie).

**Online step for Ana.** She has rated Toy Story, Nemo and Matrix. The candidates are the movies she has not rated: Terminator and Titanic. For each candidate, add its similarity to each of her movies:

| Candidate | from Toy Story | from Nemo | from Matrix | Total score |
|---|---|---|---|---|
| Terminator | 0.50 | 0.50 | 0.50 | **1.50** |
| Titanic | 0.25 | 0.67 | 0.25 | 1.17 |

**Top-1 recommendation for Ana: Terminator.** Wrong again!

**Why?** Jaccard only sees *that* Ana rated Matrix, not that she gave it one star. A movie she hated is counted as if it were a movie she chose.

**How this is handled in real systems.** The table is built from **positive signals only**: purchases, completed views, likes, or ratings of 4 and above. A one-star rating is not a "watched together" signal. Do the same here, keeping only ratings ≥ 4:

| Movie | Users who *liked* it (rating ≥ 4) |
|---|---|
| Toy Story | {Ana, Cem} |
| Nemo | {Ana, Cem} |
| Matrix | {Ben, Dee} |
| Terminator | {Ben, Dee} |
| Titanic | { } |

Now Jaccard(Toy Story, Nemo) = 2/2 = 1.0, Jaccard(Matrix, Terminator) = 1.0, and every animation–action pair is 0/4 = 0. Ana liked Toy Story and Nemo, so Terminator's score is $0 + 0 = 0$. **Terminator is no longer recommended to Ana.** (With such a tiny catalog there is nothing else left to recommend, and the system would fall back to popularity.)

The lesson: the similarity formula matters, but **which interactions you count as a positive signal matters just as much.** This is a daily decision in industry.

**Three ways to combine the neighbour scores.** The professor mentions them; they can give different lists. Suppose a user watched X and Y, with `knn_map[X] = [(0.9, P), (0.5, Q)]` and `knn_map[Y] = [(0.6, Q), (0.4, R)]`:

| Rule | P | Q | R | Order |
|---|---|---|---|---|
| total similarity (sum) | 0.9 | 0.5 + 0.6 = 1.1 | 0.4 | Q, P, R |
| maximum similarity | 0.9 | 0.6 | 0.4 | P, Q, R |
| frequency (count) | 1 | 2 | 1 | Q, then P or R |

"Sum" rewards items that are connected to *several* things the user liked, which is usually what we want.

In [14]:
items_t = list(T.columns)
J_t = pd.DataFrame([[jaccard(T, i, j) if i != j else np.nan for j in items_t] for i in items_t], index=items_t, columns=items_t)
print("Jaccard between all pairs (the kNN-map):"); print(J_t.round(2), "\n")

seen = T.loc["Ana"].dropna().index
print("Top-N scores for Ana (sum of similarities to the movies she rated):")
print(J_t.loc[seen].drop(columns=seen).sum().round(2).sort_values(ascending=False).to_string(), "\n")

L = T.where(T >= 4)                                           # keep positive signals only
liked = L.loc["Ana"].dropna().index
J_like = pd.DataFrame([[jaccard(L, i, j) if (L[i].notna() | L[j].notna()).any() and i != j else 0.0 for j in items_t] for i in items_t], index=items_t, columns=items_t)
print("Same thing using only ratings >= 4:")
print(J_like.loc[liked].drop(columns=seen).sum().round(2).to_string())

Jaccard between all pairs (the kNN-map):
            ToyStory  Nemo  Matrix  Terminator  Titanic
ToyStory         NaN  0.50    0.50        0.50     0.25
Nemo            0.50   NaN    0.50        0.50     0.67
Matrix          0.50  0.50     NaN        0.50     0.25
Terminator      0.50  0.50    0.50         NaN     0.67
Titanic         0.25  0.67    0.25        0.67      NaN 

Top-N scores for Ana (sum of similarities to the movies she rated):
Terminator    1.50
Titanic       1.17 

Same thing using only ratings >= 4:
Terminator    0.0
Titanic       0.0


### Step 4 — the cost problem and the Amazon algorithm

**Why the offline step is expensive.** To build the kNN-map we compared every pair of items. With $n$ items there are $n(n-1)/2$ pairs:

| Items $n$ | Pairs to compare |
|---|---|
| 5 | 10 |
| 9,724 (MovieLens small) | about 47 million |
| 1,000,000 | about 500 billion |

This is what $O(n^2)$ means: ten times more items → a hundred times more work. That is professor's question Q2.

**The observation behind the Amazon algorithm.** In a real catalog, almost all pairs of items have **no user in common**. A person who bought a phone case and a person who bought garden soil share nothing, so the similarity of "phone case" and "garden soil" is 0. Computing a zero is wasted work. So do not go through all pairs; go through the **users**, and only touch pairs that some user really connects.

```
AMAZON_ITEM_TO_ITEM:
    for each item i:
        for each user u who rated i:
            for each other item j that u rated:
                count[i][j] <- count[i][j] + 1        # i and j were rated together once more
        for each j that got a count:
            compute sim(i, j)
        keep the k best
```

**By hand — the neighbours of Titanic.**

- Who rated Titanic? Cem and Dee.
- What else did Cem rate? Toy Story, Nemo, Terminator → each gets +1.
- What else did Dee rate? Nemo, Matrix, Terminator → each gets +1.

| Other movie | Co-rating count with Titanic |
|---|---|
| Nemo | 2 |
| Terminator | 2 |
| Toy Story | 1 |
| Matrix | 1 |

The count is exactly the numerator of Jaccard, so the similarity follows immediately. Nemo has 3 raters, Titanic has 2:

$$J(\text{Titanic}, \text{Nemo}) = \frac{\text{count}}{n_{\text{Titanic}} + n_{\text{Nemo}} - \text{count}} = \frac{2}{2 + 3 - 2} = 0.67 \quad\checkmark$$

In our tiny dense table every movie is connected to every other, so nothing is saved. In a real sparse catalog an item is connected to a few thousand others out of a million, and the saving is enormous.

**Cost of the Amazon algorithm.** Worst case $O(n^2 m)$ ($m$ = number of users), but in practice close to $O(n\,m)$, because most users have only a handful of items, so the inner loop is short. The few users with thousands of items are usually sampled.

**Other ways around $O(n^2)$** (the professor lists them): approximate nearest-neighbour search such as LSH, which finds *almost* the best neighbours much faster; doing the computation as a matrix multiplication on a GPU; or sampling candidates.

In [15]:
def amazon_cooccurrence(item_users, user_items):
    """Linden et al. 2003: item -> its users -> their other items. Only pairs that share a user are ever touched."""
    co = {}
    for i, users in item_users.items():
        c = defaultdict(int)
        for u in users:
            for j in user_items[u]:
                if j != i: c[j] += 1
        co[i] = dict(c)
    return co

has = T.notna()
item_users_t = {i: list(has.index[has[i]]) for i in has.columns}
user_items_t = {u: list(has.columns[has.loc[u]]) for u in has.index}
co_t = amazon_cooccurrence(item_users_t, user_items_t)
n_t = has.sum()
print("co-rating counts for Titanic:", co_t["Titanic"])
print("Jaccard(Titanic, Nemo) from the counts:", round(co_t["Titanic"]["Nemo"] / (n_t["Titanic"] + n_t["Nemo"] - co_t["Titanic"]["Nemo"]), 2))

co-rating counts for Titanic: {'ToyStory': 1, 'Nemo': 2, 'Terminator': 2, 'Matrix': 1}
Jaccard(Titanic, Nemo) from the counts: 0.67


### The algorithms in general form, with their cost

```
PREDICT(u, i, k):                                    # rating prediction
    for each item j rated by u:
        s[j] <- sim(i, j)
    N <- the k items j with the largest s[j]
    if the sum of s over N is 0: return a fallback (item mean or global mean)
    return  sum over N of s[j] * r[u][j]  /  sum over N of |s[j]|

TOPN_PREDICT_SORT(u, N):                             # slow way
    for each item i NOT rated by u:
        score[i] <- PREDICT(u, i)
    return the N items with the highest score

BUILD_KNN_MAP(k):                                    # offline
    for each item i:
        for each other item j:
            compute sim(i, j) and keep it if it is among the k best so far
        knn_map[i] <- the k best (sim, j)

TOPN_KNN_MAP(u, N):                                  # online, fast way
    C <- empty table  item -> score
    for each item j that u interacted with:
        for each (sim, i) in knn_map[j]:
            if u has not interacted with i:
                C[i] <- C[i] + sim
    return the N items with the highest score in C
```

Symbols: $n$ items, $m$ users, $\bar u$ = average number of ratings per item, $|I_u|$ = number of items of user $u$.

| Operation | Cost | Why |
|---|---|---|
| one similarity `sim(i,j)` | $O(\bar u)$ | compare the two lists of users |
| `PREDICT(u,i)` | $O(\lvert I_u\rvert \cdot \bar u)$ | one similarity for each item the user rated |
| `TOPN_PREDICT_SORT` | $O(n \cdot \lvert I_u\rvert \cdot \bar u)$ | a prediction for every item in the catalog — per request |
| `BUILD_KNN_MAP` | $O(n^2 \cdot \bar u)$ time, $O(n\,k)$ memory | all pairs, but only $k$ neighbours stored per item — once, offline |
| `TOPN_KNN_MAP` | $O(\lvert I_u\rvert \cdot k)$ | only table lookups; **does not depend on the catalog size** |
| Amazon algorithm | worst $O(n^2 m)$, in practice $\approx O(n\,m)$ | only pairs that share a user |

The key design idea, which you will see in every real recommender: **move the expensive work offline, keep the online step to simple lookups.**

## 2.C Professor's code — line by line

### Data structures

The professor builds two dictionaries so that a lookup goes from $O(|R|)$ (filtering a DataFrame) to $O(1)$:

- `rating_map["userId_movieId"] -> rating` — e.g. `rating_map["1_101"] = 5.0`
- `user_ratings_map[movieId] -> (userids, ratings)` — the "item vector" in sparse form

He builds them with a loop + `.iloc` / `.query` (slow: the whole DataFrame is scanned for every item → $O(n\cdot|R|)$). The same output in a single pass:

In [16]:
def build_maps(df):
    r_map  = {f"{u}_{i}": r for u, i, r in zip(df.userId, df.movieId, df.rating)}
    ur_map = {i: (g.userId.to_numpy(), g.rating.to_numpy()) for i, g in df.groupby("movieId")}
    return r_map, ur_map

rating_map, user_ratings_map = build_maps(ratings)
print(rating_map["1_101"], "|", user_ratings_map[10][0][:5], user_ratings_map[10][1][:5])

5.0 | [ 6  8 11 19 21] [3. 2. 3. 2. 5.]


### `calc_sim` and `NNCF_based_rating_prediction` (the professor's code, logic unchanged)

In [17]:
def calc_sim(i, j, metric, ur_map):
    users_rated_i, ratings_i = ur_map[i]
    users_rated_j, ratings_j = ur_map[j]
    if metric == "Jaccard":                                   # ignores the rating VALUES
        inter = len(set(users_rated_i).intersection(users_rated_j))
        union = len(set(users_rated_i).union(users_rated_j))
        return 0 if union == 0 else inter / union
    elif metric == "Cosine":
        # common users + their positions in each array
        inter, ind1, ind2 = np.intersect1d(users_rated_i, users_rated_j, return_indices=True)
        if len(inter) == 0:
            return 0
        norm_i = np.linalg.norm(ratings_i[ind1])              # norms over CO-RATED users only  <-- key detail
        norm_j = np.linalg.norm(ratings_j[ind2])
        if norm_i == 0 or norm_j == 0:
            return 0
        return np.dot(ratings_i[ind1], ratings_j[ind2]) / (norm_i * norm_j)

def NNCF_based_rating_prediction(u, i, metric, ratings_df, r_map, ur_map):
    r, sum_sim = 0, 0
    movies_u = ratings_df[ratings_df["userId"] == u].movieId  # all items rated by u (no top-k!)
    for j in movies_u:
        sim = calc_sim(i, j, metric, ur_map)
        r += sim * r_map[f"{u}_{j}"]                           # numerator: sim * rating
        sum_sim += sim                                         # denominator: sum of sims
    return 0 if sum_sim == 0 else r / sum_sim

**Reading `calc_sim`.**

- `ur_map[i]` returns two aligned arrays for item $i$: the users who rated it and their ratings. This is the "column" of our hand table, stored without the empty cells.
- **Jaccard branch:** turns the two user arrays into sets and divides the size of the intersection by the size of the union. Exactly Step 1a.
- **Cosine branch:** `np.intersect1d(..., return_indices=True)` finds the co-rated users and *where* they are in each array, so that `ratings_i[ind1]` and `ratings_j[ind2]` are the two short vectors we wrote by hand in Step 1b. Then dot product divided by the two lengths.

**Reading `NNCF_based_rating_prediction`.** It is the `PREDICT` formula: loop over all movies the user rated, accumulate `sim * rating` in the numerator and `sim` in the denominator. There is no $k$ (all rated movies are used) and the fallback when nothing is similar is 0, which is a poor prediction on a 1–5 scale.

`np.intersect1d(..., return_indices=True)` returns three things: the common users, and their **indices** in the first and second array — so that the ratings of the two items can be aligned on the same users. The professor's own example:

In [18]:
users_i, ratings_i = np.array([1, 3, 6, 9, 12, 15]), np.array([4, 3, 2, 4, 2, 5])
users_j, ratings_j = np.array([1, 6, 8, 12]),        np.array([2, 4, 2, 5])
inter, ind1, ind2 = np.intersect1d(users_i, users_j, return_indices=True)
print("common users:", inter, "| idx in i:", ind1, "| idx in j:", ind2)
print("aligned ratings:", ratings_i[ind1], ratings_j[ind2])
print("cosine =", 26, "/ (sqrt(24)*sqrt(45)) =", round(26 / (np.sqrt(24) * np.sqrt(45)), 3))

common users: [ 1  6 12] | idx in i: [0 2 4] | idx in j: [0 1 3]
aligned ratings: [4 2 2] [2 4 5]
cosine = 26 / (sqrt(24)*sqrt(45)) = 0.791


### The professor's question: "How can you explain the difference between using Cosine vs. Jaccard?"

In the professor's output Jaccard (MAE ≈ 0.74) came out **better** than Cosine (≈ 0.79), even though Jaccard does not look at the rating values at all. First let us repeat the experiment (with a larger test set, because his 100 samples are very noisy) and keep the distribution of the similarities at the same time:

In [19]:
Xtr_p, Xte_p = train_test_split(ratings, test_size=300, random_state=SEED)
train_r_map, train_ur_map = build_maps(Xtr_p)                 # maps from TRAIN only -> no leakage
mu_p = Xtr_p.rating.mean()
user_mean_p = Xtr_p.groupby("userId").rating.mean()

errs, sims = defaultdict(list), defaultdict(list)
for u, i, r in zip(Xte_p.userId, Xte_p.movieId, Xte_p.rating):
    for metric in ["Cosine", "Jaccard"]:
        try:    p = NNCF_based_rating_prediction(u, i, metric, Xtr_p, train_r_map, train_ur_map)
        except KeyError: p = mu_p                             # item never seen in train -> fallback
        errs[metric].append(abs(r - p))
    errs["Global average"].append(abs(r - mu_p))
    errs["User average"].append(abs(r - user_mean_p.get(u, mu_p)))
    if i in train_ur_map:                                     # keep the similarity distributions
        for j in Xtr_p[Xtr_p.userId == u].movieId.head(40):
            for metric in ["Cosine", "Jaccard"]:
                sims[metric].append(calc_sim(i, j, metric, train_ur_map))

print(pd.Series({k: np.mean(v) for k, v in errs.items()}, name="MAE").round(4).to_string())
sim_stats = pd.DataFrame({m: pd.Series(v)[pd.Series(v) > 0].describe()[["mean", "std", "25%", "50%", "75%"]] for m, v in sims.items()})
print("\nDistribution of the NON-ZERO similarities:"); print(sim_stats.round(3))
print("\nshare of non-zero cosine sims that are > 0.9 :", round(float((pd.Series(sims['Cosine'])[pd.Series(sims['Cosine'])>0] > 0.9).mean()), 3))

Cosine            0.7579
Jaccard           0.7203
Global average    0.8184
User average      0.7554

Distribution of the NON-ZERO similarities:
      Cosine  Jaccard
mean   0.961    0.124
std    0.038    0.092
25%    0.948    0.054
50%    0.968    0.104
75%    0.986    0.171

share of non-zero cosine sims that are > 0.9 : 0.938


**Answer (know this for the exam):**

1. **The professor's cosine is computed only over common users and all ratings are positive.** So almost every similarity comes out close to 1 (table above: high mean, very small spread). When all weights are nearly equal, the weighted average effectively becomes the **plain average of the user's ratings**; that is why the cosine MAE is almost identical to "User average". Cosine has added no information about "which items are related to $i$".
2. **It ignores the number of common users.** Two items with a single common user get cosine = 1.0 — the largest possible weight, from the least reliable evidence.
3. **Jaccard** throws away the rating values but measures the **strength of co-occurrence**: items watched by the same people get a high weight and the rest get a weight near zero. Its distribution is wide, so it really discriminates between neighbours. "Who watched this" is by itself a strong taste signal.
4. **Fixes for cosine:** mean-centering (adjusted cosine / Pearson), computing the norm over the whole vector (missing = 0), or **significance weighting / shrinkage**: $\text{sim}' = \frac{n_{ij}}{n_{ij}+\beta}\,\text{sim}$ where $n_{ij}$ is the number of common users.

In section 2.D below you will see that with these fixes cosine overtakes Jaccard.

**Connect this to the hand example.** It is the same thing we saw in Step 1b on the tiny table: raw cosine gave 0.65, 0.69 and 0.98, all high and all positive, even for two movies that are opposites. On real data the effect is stronger still, because many pairs of movies share only one or two users, and with one common user the cosine is exactly 1.

### Top-N — Predict & Sort (the professor's code)

```python
def top_N_pred_sort(N, u):
    preds = pd.Series([], dtype='float')
    all_items = set(ratings['movieId'].unique())
    rated_by_user = set(ratings[ratings["userId"]==1].movieId.unique())   # <-- BUG: hard-coded user 1, should be u
    not_rated_by_user = all_items - rated_by_user
    sample_movies = np.random.choice(list(not_rated_by_user), 500)        # only 500 sampled items, because it is too slow
    for m in tqdm(sample_movies):
        preds[m] = NNCF_based_rating_prediction(u, m, "Jaccard", ratings, rating_map, user_ratings_map)
    return preds.sort_values(ascending=False)[:N]
```

- **Bug:** `userId==1` instead of `u`; for any user other than 1, the "unseen" items are computed wrongly.
- The professor himself had to score only 500 random items — in effect showing that this algorithm does not scale.

**The professor's question in "Efficiency Issues": "Can you suggest a solution?"** (to avoid predicting the rating of every item)

> Do not score the whole catalog. Use a two-stage design: (1) **candidate generation** — cheaply select a few hundred candidates, e.g. the precomputed k nearest neighbours of the items the user already interacted with (kNN-map), or approximate nearest-neighbour search (LSH/HNSW) in an embedding space; (2) score/rank only those candidates. In addition, **precompute item–item similarities offline** and store only the top-k per item, so that no similarity is computed at request time.

### Building the kNN-map (the professor's code)

```python
def build_knn_map(movies, K=30, max_candidates=500):
    knn_map = {}
    movie_ids = ratings['movieId'].unique()
    for i in tqdm(movie_ids):
        sorted_list = []                      # bounded sorted list of (sim, movieId), ascending
        knn_map[i] = sorted_list
        other_movies = movie_ids[movie_ids != i]
        candidate_list = np.random.choice(other_movies, max_candidates, replace=False)   # sampling for speed
        for j in candidate_list:
            sim = calc_sim(i, j, "Jaccard", user_ratings_map)
            if len(sorted_list) < K:
                bisect.insort(sorted_list, (sim, j))           # keep sorted
            elif sim > sorted_list[0][0]:                      # better than the current worst?
                bisect.insort(sorted_list, (sim, j))
                sorted_list.pop(0)                             # drop the smallest
    return knn_map
```

**What it does:** for every item it keeps a sorted list of length at most $K$. `sorted_list[0]` is always the **least similar** current member; if the new candidate is better than it, the candidate goes in and the smallest is removed. This is the classic **"top-K with $O(K)$ memory"** pattern.

**Notes:**

- `max_candidates=500` means each item is compared with only 500 **random** items, not all of them → the neighbours are approximate and the true neighbour is often never seen. The cost goes from $O(n^2)$ to $O(500\,n)$.
- `bisect.insort` finds the insertion point by binary search in $O(\log K)$, but the insertion into the list itself and `pop(0)` are both $O(K)$. The more appropriate structure is a **min-heap** (`heapq.heappushpop` → $O(\log K)$). Total cost of selecting the top-K from $c$ candidates: $O(c \log K)$ versus $O(c \log c)$ for a full sort.

The heap version for one item, with **all** candidates (no sampling):

In [20]:
def knn_for_item(i, ur_map, K=10, metric="Jaccard"):
    heap = []                                                 # min-heap of (sim, item): heap[0] is the worst kept
    for j in ur_map:
        if j == i: continue
        s = calc_sim(i, j, metric, ur_map)
        if len(heap) < K:    heapq.heappush(heap, (s, j))
        elif s > heap[0][0]: heapq.heappushpop(heap, (s, j))  # push new, pop smallest: O(log K)
    return sorted(heap, reverse=True)

t0 = time.time()
nbrs = knn_for_item(1, user_ratings_map)                      # movieId 1 = Toy Story
print(f"one item vs all {n_items} items: {time.time()-t0:.2f}s  ->  full map ~ {(time.time()-t0)*n_items/60:.0f} min in pure Python\n")
for s, j in nbrs[:6]: print(f"  {s:.3f}  {title[j]}")

one item vs all 9724 items: 0.29s  ->  full map ~ 47 min in pure Python

  0.414  Independence Day (a.k.a. ID4) (1996)
  0.411  Jurassic Park (1993)
  0.404  Star Wars: Episode IV - A New Hope (1977)
  0.395  Forrest Gump (1994)
  0.393  Star Wars: Episode VI - Return of the Jedi (1983)
  0.391  Mission: Impossible (1996)


The neighbours make sense, but look at the time: brute force in pure Python takes tens of minutes for the whole map. That is the reason for the professor's `max_candidates`, and the reason for section 2.D.

### Amazon algorithm (the professor's code)

```python
users  = ratings.query("movieId == @i").userId.unique()        # users who rated i
movies = ratings.query("userId in @users").movieId.unique()    # every item those users rated = candidates
movies = np.random.choice(movies, 500)                         # speed-up, NOT part of the Amazon algorithm
for j in movies: ...same bounded sorted list...
```

The only difference from `build_knn_map`: instead of "500 random items from the whole catalog", the candidates come from **the items that share at least one user with $i$**; everything else has similarity zero and checking it is a waste of time. The professor himself says it does not help much on MovieLens (because it is relatively dense and popular items are co-rated with almost everything), but on very sparse data such as Amazon's the difference is huge.

(Two small points: `np.random.choice(movies, 500)` samples with replacement, so it produces duplicate candidates and can put the same neighbour in the list twice; and the variable `movies` overwrites the function's input argument.)

The clean implementation of the core of the algorithm is the `amazon_cooccurrence` function in section 2.B.

### Top-N with the kNN-map (the professor's code) — and an important bug

```python
def add_sims_and_sort(l):
    li = []
    it = itertools.groupby(l, operator.itemgetter(1))        # group by movieId
    for key, subiter in it:
        li.append((key, sum(item[0] for item in subiter)))   # sum the similarities per movie
    return sorted(li, key=itemgetter(1), reverse=True)

def top_N_knn_map(ratings, N, u):
    C = []
    for m in ratings.query("userId == @u").movieId:
        C = C + knn_map[m]                                    # concatenate neighbour lists
    return add_sims_and_sort(C)[:N]
```

Goal: "highest total similarity" — the sum of each candidate's similarities over all of the user's items.

**Bug:** `itertools.groupby` only groups **consecutive** elements with the same key, so the input must already be sorted by key. Here `C` is not sorted by movieId, so the similarities of a movie that came from several neighbours are **not summed**, and the movie stays in the list several times as separate entries. In effect the professor's code computes "max similarity", not "total similarity". The sign of this in his own output: all scores are below 1.

In [21]:
C = [(0.9, "P"), (0.5, "Q"), (0.4, "R"), (0.6, "Q")]      # concatenated neighbour lists; the two Q entries are NOT adjacent

def add_sims_and_sort_prof(l):
    li = [(key, sum(x[0] for x in sub)) for key, sub in itertools.groupby(l, operator.itemgetter(1))]
    return sorted(li, key=operator.itemgetter(1), reverse=True)

def add_sims_and_sort_fixed(l):
    acc = defaultdict(float)
    for s, item in l: acc[item] += s                          # dict accumulation: order does not matter
    return sorted(acc.items(), key=operator.itemgetter(1), reverse=True)

print("professor:", add_sims_and_sort_prof(C))
print("fixed    :", add_sims_and_sort_fixed(C))

professor: [('P', 0.9), ('Q', 0.6), ('Q', 0.5), ('R', 0.4)]
fixed    : [('Q', 1.1), ('P', 0.9), ('R', 0.4)]


**Three more flaws in the same function** (each could be a "what is wrong with this code" question):

1. **Watched movies are not removed** — it can recommend something the user has already seen.
2. **Leakage in the evaluation:** `knn_map` is built on **all** of `ratings`, and the test set is split off afterwards; so the similarities have seen the test ratings. The kNN-map should be built from train only.
3. The comment says `ratings >= 4` but the code is `rating > 4` (that is, only 4.5 and 5).

### Hit Ratio (the professor's code)

Here the protocol differs from Part 1: **no negative sampling**. For every $(u,i)$ in the test set a real top-N list is built and we check whether $i$ is in it.
$$\text{Hit Ratio} = \frac{\#\text{hits}}{|T|}$$
The professor got 0.165 with $N=100$. Note that this number is **not comparable** with the HRs of Part 1 (there the rank was among 100 candidates, here it is among the whole catalog).

### Why this code is fine for a class but not how it is done at work

| What the professor does | Why it is a problem in practice | What is done instead |
|---|---|---|
| Dictionaries with string keys such as `"1_101"` | Slow to build, heavy in memory, and cannot be used for fast math | One **sparse matrix** (users × items); a lookup is an index |
| `ratings.query("movieId == @i")` inside a loop over all items | Scans the whole table once per item → $O(n \cdot \lvert R\rvert)$ just to prepare the data | One `groupby`, or the sparse matrix |
| A Python double loop calling `calc_sim` for each pair | Tens of minutes for 9,724 movies; impossible for a real catalog | **Matrix multiplication**: all similarities at once, in compiled code |
| Comparing each item with 500 *random* items to save time | The true nearest neighbours are usually not among the 500, so the neighbour lists are poor | Exact computation with sparse products, or a proper approximate index (FAISS, HNSW) |
| Raw cosine over co-rated users | Almost every similarity is near 1 (see the question above) | Cosine over full vectors on implicit data, adjusted cosine with shrinkage, or learned embeddings |
| Sorted list + `pop(0)` to keep the top-K | Insertion and `pop(0)` are $O(K)$ | A heap, or `np.argpartition` on a whole row |
| Similarities computed while the request is being served | Far too slow for an online system | Precompute offline; online only lookups |
| kNN-map built on all data, then evaluated | Leakage: the test ratings influenced the similarities | Build from the training data only |

The ideas in his notebook are exactly the right ones. Only the way they are executed belongs to the classroom. Section 2.D does the same algorithm the industrial way.

## 2.D The industry version

The main idea: **the entire item-kNN algorithm is a few sparse matrix products.** If $X$ is the user×item matrix and its columns are L2-normalized:

$$S = X^\top X \quad\Rightarrow\quad S_{ij} = \cos(i,j) \qquad\qquad \text{scores} = X \cdot S_k^\top$$

and a sparse product automatically touches only co-rated pairs — that is, **the Amazon algorithm itself, but in C**.

### Step 1 — the sparse matrix

In [22]:
user_ids, item_ids = np.sort(ratings.userId.unique()), np.sort(ratings.movieId.unique())
u2idx = pd.Series(np.arange(n_users), index=user_ids)        # raw id -> contiguous index (always needed in practice)
i2idx = pd.Series(np.arange(n_items), index=item_ids)

def to_csr(df, binary=False):
    v = np.ones(len(df)) if binary else df.rating.to_numpy(float)
    return csr_matrix((v, (u2idx[df.userId].to_numpy(), i2idx[df.movieId].to_numpy())), shape=(n_users, n_items))

X_all = to_csr(ratings)
print(repr(X_all))
print(f"memory: sparse {X_all.data.nbytes/1e6:.1f} MB  vs  dense {n_users*n_items*8/1e6:.1f} MB")

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 100836 stored elements and shape (610, 9724)>
memory: sparse 0.8 MB  vs  dense 47.5 MB


**Why a sparse matrix.** The full table has 610 × 9,724 ≈ 5.9 million cells, but only about 100,000 are filled. A sparse matrix stores only the filled cells. At real scale (100 million users × 1 million items) the full table could not be stored at all; the sparse one is routine.

**Why the id mapping.** Movie ids in the data go up to 193,609 with gaps. Matrices need positions 0, 1, 2, … so every real pipeline starts by mapping raw ids to consecutive indices and keeping the mapping to translate results back.

### Step 2 — top-k similarity for all items (chunked)

We do not keep the full $n \times n$ matrix; we compute it chunk by chunk and keep only the top-k of each row (memory $O(nk)$).

In [23]:
def topk_item_sim(X, k=30, metric="cosine", center=None, shrink=0, chunk=1000):
    """X: users x items (CSR). Returns sparse items x items matrix, row i = top-k neighbours of item i.
    metric : 'cosine' (missing = 0, full-vector norms) or 'jaccard'
    center : None | 'user'  -> subtract the user's mean from each rating first (adjusted cosine)
    shrink : beta in  sim * n_ij / (n_ij + beta)   (significance weighting)"""
    X = X.tocsr().astype(float)
    B = X.copy(); B.data[:] = 1.0                             # binary "who rated what"
    if center == "user":
        X = X.copy()
        means = np.asarray(X.sum(1)).ravel() / np.maximum(np.diff(X.indptr), 1)
        X.data -= np.repeat(means, np.diff(X.indptr))
    Bi, cnt = B.T.tocsr(), np.asarray(B.sum(0)).ravel()       # items x users, #ratings per item
    Xi = Bi if metric == "jaccard" else normalize(X.T.tocsr())
    rows, cols, vals = [], [], []
    for s in range(0, X.shape[1], chunk):
        e   = min(s + chunk, X.shape[1])
        co  = (Bi[s:e] @ Bi.T).toarray()                      # co-rating counts n_ij  (sparse product = Amazon trick)
        sim = co / np.maximum(cnt[s:e, None] + cnt[None, :] - co, 1) if metric == "jaccard" else (Xi[s:e] @ Xi.T).toarray()
        if shrink: sim = sim * co / (co + shrink)
        sim[np.arange(e - s), np.arange(s, e)] = 0            # no self-similarity
        idx = np.argpartition(-sim, k, axis=1)[:, :k]         # top-k per row in O(n), no full sort
        rows.append(np.repeat(np.arange(s, e), k)); cols.append(idx.ravel()); vals.append(np.take_along_axis(sim, idx, 1).ravel())
    S = csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(X.shape[1],) * 2)
    S.data[S.data < 0] = 0; S.eliminate_zeros()               # keep positive neighbours only
    return S

t0 = time.time()
S_demo = topk_item_sim(to_csr(ratings, binary=True), k=10, metric="jaccard")
print(f"exact Jaccard top-10 for ALL {n_items} items: {time.time()-t0:.1f}s   (pure-Python estimate above: tens of minutes)\n")
row = S_demo[i2idx[1]]
for j, s in sorted(zip(row.indices, row.data), key=lambda t: -t[1])[:6]: print(f"  {s:.3f}  {title[item_ids[j]]}")

exact Jaccard top-10 for ALL 9724 items: 2.6s   (pure-Python estimate above: tens of minutes)

  0.414  Independence Day (a.k.a. ID4) (1996)
  0.411  Jurassic Park (1993)
  0.404  Star Wars: Episode IV - A New Hope (1977)
  0.395  Forrest Gump (1994)
  0.393  Star Wars: Episode VI - Return of the Jedi (1983)
  0.391  Mission: Impossible (1996)


The same neighbours as the pure-Python version, but for the **whole catalog** in a few seconds and **exact** (no sampling).

### Step 3 — Rating prediction and comparison of the similarities

$$\text{num} = X\,S_k^\top,\qquad \text{den} = B\,S_k^\top,\qquad \hat R = \text{num} / \text{den}$$

In [24]:
def knn_predict(Xtr, S, tu, ti, fallback, item_mean=None):
    """r_hat(u,i) = sum_j S[i,j] r_uj / sum_j S[i,j]  over j in (top-k of i) ∩ (rated by u). Optional item-mean centering."""
    B = Xtr.copy(); B.data[:] = 1.0
    V = Xtr
    if item_mean is not None:
        V = Xtr.copy(); V.data -= item_mean[V.indices]        # r_uj - mean_j
    num = np.asarray((V @ S.T)[tu, ti]).ravel()
    den = np.asarray((B @ S.T)[tu, ti]).ravel()
    ok  = den > 1e-9
    pred = fallback.copy()
    pred[ok] = num[ok] / den[ok] + (item_mean[ti][ok] if item_mean is not None else 0)
    return np.clip(pred, 0.5, 5), ok.mean()

Xtr = to_csr(X_train)                                         # same 90/10 split as Part 1
tu, ti, y = u2idx[X_test.userId].to_numpy(), i2idx[X_test.movieId].to_numpy(), X_test.rating.to_numpy()
cnt_i  = np.diff(Xtr.tocsc().indptr)
imean  = np.where(cnt_i > 0, np.asarray(Xtr.sum(0)).ravel() / np.maximum(cnt_i, 1), mu)
fb     = preds["Bias: mu+b_u+b_i"].astype(float)              # fallback when no neighbour is available

K_PRED = 40
variants = {
    "Jaccard":                                dict(metric="jaccard"),
    "Cosine (missing=0)":                     dict(metric="cosine"),
    "Adjusted cosine + shrink":               dict(metric="cosine", center="user", shrink=20),
}
rows = {}
for name, kw in variants.items():
    S = topk_item_sim(Xtr, k=K_PRED, **kw)
    p, cov = knn_predict(Xtr, S, tu, ti, fb)
    rows[f"item-kNN | {name}"] = {"MAE": mae(y, p), "RMSE": rmse(y, p), "coverage": cov}
p, cov = knn_predict(Xtr, S, tu, ti, fb, item_mean=imean)     # last S = adjusted cosine
rows["item-kNN | Adjusted cosine + shrink + mean-centered prediction"] = {"MAE": mae(y, p), "RMSE": rmse(y, p), "coverage": cov}
knn_table = pd.concat([rating_table.assign(coverage=1.0).loc[["Global mean", "User mean", "Bias: mu+b_u+b_i"]], pd.DataFrame(rows).T])
knn_table

,MAE,RMSE,coverage
Global mean,0.8351,1.0506,1.0000
User mean,0.7432,0.9563,1.0000
Bias: mu+b_u+b_i,0.6796,0.8861,1.0000
item-kNN | Jaccard,0.6767,0.9036,0.8460
item-kNN | Cosine (missing=0),0.6775,0.9079,0.8148
item-kNN | Adjusted cosine + shrink,0.6618,0.8930,0.9111
item-kNN | Adjusted cosine + shrink + mean-centered prediction,0.6602,0.8795,0.9111


`coverage` = the share of the test set that had at least one neighbour (the rest fall back to the bias baseline).

**What to take from the table:**

- The "smarter" the similarity gets (mean-centering + shrinkage), and once the prediction is mean-centered too, the lower the error. This is the path section 2.B described.
- **The bias baseline is a tough competitor.** kNN is slightly better in MAE, but in RMSE only the best variant reaches it; the best kNN versions (such as `KNNBaseline` below) actually put kNN **on top of the baseline's residual**.

### Surprise — the same algorithm in three lines

In [25]:
from surprise import KNNBasic, KNNWithMeans, KNNBaseline

algos = {
    "KNNBasic  (cosine)":            KNNBasic(k=40, sim_options={"name": "cosine", "user_based": False}, verbose=False),
    "KNNWithMeans (pearson)":        KNNWithMeans(k=40, sim_options={"name": "pearson", "user_based": False}, verbose=False),
    "KNNBaseline (pearson_baseline)": KNNBaseline(k=40, sim_options={"name": "pearson_baseline", "user_based": False, "shrinkage": 100}, verbose=False),
}
for name, algo in algos.items():
    cv = cross_validate(algo, sdata, measures=["MAE", "RMSE"], cv=3, verbose=False)
    print(f"{name:32s} MAE={cv['test_mae'].mean():.4f}  RMSE={cv['test_rmse'].mean():.4f}")

KNNBasic  (cosine)               MAE=0.7603  RMSE=0.9761


KNNWithMeans (pearson)           MAE=0.6948  RMSE=0.9131


KNNBaseline (pearson_baseline)   MAE=0.6594  RMSE=0.8643


`user_based: False` means item-based. `KNNBasic` with cosine uses the same similarity as the professor (raw cosine over co-rated users) — and you can see it is the worst. `pearson_baseline` with `shrinkage` is exactly the set of fixes I listed in the answer to the professor's question.

### Step 4 — Top-N with the kNN-map, in matrix form

$$\text{score}(u, i) = \sum_{j \in I_u} S_k[j, i] \quad\Longleftrightarrow\quad \text{Scores} = B \cdot S_k$$

This is exactly the professor's "highest total similarity" (the correct version, without the bug). Evaluation: the leave-one-out split from Part 1, but this time with **full ranking** over the whole catalog (no negative sampling) and the kNN-map built **from train only**.

In [26]:
def hit_ratio_full(scores, B_train, test, Ks=(10, 50, 100)):
    """Full-catalog ranking. scores: users x items (dense). Seen items are masked out."""
    sc = np.array(scores, dtype=float)
    sc[B_train.nonzero()] = -np.inf                           # never recommend what the user already has
    tu, ti = u2idx[test.userId].to_numpy(), i2idx[test.movieId].to_numpy()
    rank = (sc[tu] > sc[tu, ti][:, None]).sum(1)              # rank of the held-out item for each test user
    return {f"HR@{k}": float((rank < k).mean()) for k in Ks}

B_tr = to_csr(tr, binary=True)                                # tr / te: leave-one-out split from Part 1
pop_scores = np.tile(np.asarray(B_tr.sum(0)).ravel(), (n_users, 1))

S_jac = topk_item_sim(B_tr, k=30, metric="jaccard")
S_cos = topk_item_sim(B_tr, k=30, metric="cosine")            # cosine on implicit 0/1 data
topn_full = {
    "Popularity":                 hit_ratio_full(pop_scores, B_tr, te),
    "item-kNN Jaccard (k=30)":    hit_ratio_full((B_tr @ S_jac).toarray(), B_tr, te),
    "item-kNN Cosine  (k=30)":    hit_ratio_full((B_tr @ S_cos).toarray(), B_tr, te),
}
pd.DataFrame(topn_full).T

,HR@10,HR@50,HR@100
Popularity,0.1571,0.3281,0.4206
item-kNN Jaccard (k=30),0.2356,0.4468,0.5515
item-kNN Cosine (k=30),0.2461,0.4538,0.5462


Now personalized kNN **beats** popularity under full ranking — something that was not visible from the professor's 0.165 (with sampling and the bug).

The dictionary version of the same kNN-map (the professor's form) for one user, with correct aggregation and with watched items removed:

In [27]:
knn_map = {item_ids[i]: [(s, item_ids[j]) for j, s in zip(S_cos[i].indices, S_cos[i].data)] for i in range(n_items)}

def top_N_knn_map(train_df, N, u, agg="sum"):
    seen, acc = set(train_df.loc[train_df.userId == u, "movieId"]), defaultdict(float)
    for m in seen:
        for s, j in knn_map.get(m, []):
            if j in seen: continue                            # filter already-watched items
            acc[j] = acc[j] + s if agg == "sum" else max(acc[j], s) if agg == "max" else acc[j] + 1
    return heapq.nlargest(N, acc.items(), key=operator.itemgetter(1))

U = 100
print("User", U, "liked:", "; ".join(title[m] for m in tr[(tr.userId == U) & (tr.rating >= 5)].movieId.head(5)))
for m, s in top_N_knn_map(tr, 8, U): print(f"  {s:6.2f}  {title[m]}")

User 100 liked: Top Gun (1986); Christmas Vacation (National Lampoon's Christmas Vacation) (1989); Officer and a Gentleman, An (1982); Sweet Home Alabama (2002)
   17.48  Indiana Jones and the Temple of Doom (1984)
   15.97  Big (1988)
   15.74  Ferris Bueller's Day Off (1986)
   12.12  Monty Python and the Holy Grail (1975)
   10.87  Who Framed Roger Rabbit? (1988)
   10.76  E.T. the Extra-Terrestrial (1982)
   10.63  Beetlejuice (1988)
   10.12  Star Wars: Episode VI - Return of the Jedi (1983)


### `implicit` — the library that is really used in production

`implicit` (C++/Cython, multi-threaded, optional GPU) is built for **implicit feedback and Top-N**. Its item-kNN is this same algorithm; it also has weighted variants (`TfidfRecommender`, `BM25Recommender`) that should be tuned and compared on your own data.

In [28]:
from implicit.nearest_neighbours import CosineRecommender

def implicit_scores(model, B_train, N=100):
    """Turn implicit's top-N output into a users x items score matrix so that we can reuse hit_ratio_full."""
    ids, sc = model.recommend(np.arange(n_users), B_train, N=N, filter_already_liked_items=True)
    out = np.full((n_users, n_items), -1e9)
    np.put_along_axis(out, ids, sc, axis=1)
    return out

knn_model = CosineRecommender(K=30)
knn_model.fit(B_tr, show_progress=False)                      # user x item CSR
topn_full["implicit CosineRecommender (K=30)"] = hit_ratio_full(implicit_scores(knn_model, B_tr), B_tr, te)
pd.DataFrame(topn_full).T

,HR@10,HR@50,HR@100
Popularity,0.1571,0.3281,0.4206
item-kNN Jaccard (k=30),0.2356,0.4468,0.5515
item-kNN Cosine (k=30),0.2461,0.4538,0.5462
implicit CosineRecommender (K=30),0.2426,0.4538,0.5462


The `implicit` number is almost identical to our hand-written implementation — it is the same algorithm, only faster and production-ready.

In [29]:
# "Because you watched ..." : similar items in one line
ids, sc = knn_model.similar_items(int(i2idx[1]), N=6)
print("Similar to:", title[1]); [print(f"  {s:.3f}  {title[item_ids[j]]}") for j, s in zip(ids[1:], sc[1:])];

Similar to: Toy Story (1995)
  0.578  Star Wars: Episode IV - A New Hope (1977)
  0.574  Independence Day (a.k.a. ID4) (1996)
  0.574  Forrest Gump (1994)
  0.574  Jurassic Park (1993)
  0.565  Star Wars: Episode VI - Return of the Jedi (1983)


### When the catalog has millions of items: ANN with FAISS

All the code above is **exact** and $O(n^2)$. For millions of items, each item's vector (usually a compact embedding, not the raw user vector) is put into an **Approximate Nearest Neighbor index**: FAISS (Meta), ScaNN (Google), HNSW (`hnswlib`), Annoy (Spotify). This is the "LSH / approximation" family the professor mentioned. Search goes from $O(n)$ to roughly $O(\log n)$, with a small error.

In [30]:
import faiss
item_vecs = normalize(B_tr.T.tocsr()).toarray().astype("float32")    # demo: raw item vectors (610-d). Real systems: 64-256-d embeddings
index = faiss.IndexHNSWFlat(item_vecs.shape[1], 32, faiss.METRIC_INNER_PRODUCT)   # graph-based ANN index
index.add(item_vecs)
sims_, nbrs_ = index.search(item_vecs[[i2idx[1]]], 6)                # query = Toy Story
print("FAISS HNSW neighbours of:", title[1]); [print(f"  {s:.3f}  {title[item_ids[j]]}") for s, j in zip(sims_[0][1:], nbrs_[0][1:])];

FAISS HNSW neighbours of: Toy Story (1995)
  0.578  Star Wars: Episode IV - A New Hope (1977)
  0.574  Independence Day (a.k.a. ID4) (1996)
  0.574  Forrest Gump (1994)
  0.574  Jurassic Park (1993)
  0.565  Star Wars: Episode VI - Return of the Jedi (1983)


### Industry summary of Part 2

| Need | Tool |
|---|---|
| Rating prediction, teaching and prototyping | `scikit-surprise` |
| Top-N on implicit data, production at medium scale | `implicit` |
| Very large (distributed) data | Spark MLlib (ALS), or SQL/Spark jobs for co-occurrence |
| Nearest-neighbour search at scale | FAISS, ScaNN, HNSW |
| Architecture | offline: build similarities / index (batch) → online: lookup + ranking |

## 2.E Exam — kNN CF

### Worked solution of the professor's exercise Q4 (do this one on paper)

| | Item A | Item B | Item C | Item D |
|---|---|---|---|---|
| **User 1** | 5 | 3 | ? | 1 |
| **User 2** | 4 | ? | 4 | 2 |
| **User 3** | 1 | 1 | ? | 5 |
| **User 4** | 4 | 2 | 3 | ? |

**Task 1 — Jaccard(A, D).**

1. Users who rated A: {1, 2, 3, 4}. Users who rated D: {1, 2, 3}.
2. Rated both: {1, 2, 3} → 3. Rated at least one: {1, 2, 3, 4} → 4.
3. $J = 3/4 = 0.75$.

**Task 2 — Cosine(A, D)**, over the users who rated both (1, 2, 3).

1. $\vec A = [5, 4, 1]$, $\vec D = [1, 2, 5]$.
2. Dot product: $5\times1 + 4\times2 + 1\times5 = 18$.
3. Lengths: $\sqrt{25+16+1} = \sqrt{42} = 6.48$ and $\sqrt{1+4+25} = \sqrt{30} = 5.48$.
4. $\cos = 18 / (6.48 \times 5.48) = 18 / 35.50 = 0.507$.

**Task 3 — predict User 2's rating of Item B**, $k=2$, Jaccard, given $J(B,A) = 0.75$ and $J(B,D) = 0.5$.

1. User 2 rated A = 4 and D = 2. These are the two neighbours.
2. Numerator: $0.75\times4 + 0.5\times2 = 3.0 + 1.0 = 4.0$.
3. Denominator: $0.75 + 0.5 = 1.25$.
4. $\hat r = 4.0 / 1.25 = 3.2$.

**Extra — adjusted cosine(A, D)**, in case the exam asks for the mean-centered version.

1. User means: User 1: $(5+3+1)/3 = 3.00$; User 2: $(4+4+2)/3 = 3.33$; User 3: $(1+1+5)/3 = 2.33$.
2. Centered A: $[5-3,\ 4-3.33,\ 1-2.33] = [2.00,\ 0.67,\ -1.33]$. Centered D: $[1-3,\ 2-3.33,\ 5-2.33] = [-2.00,\ -1.33,\ 2.67]$.
3. Dot: $-4.00 - 0.89 - 3.56 = -8.44$. Lengths: $2.49$ and $3.59$.
4. Similarity $= -8.44 / (2.49 \times 3.59) = -0.943$.

A and D are opposites (who likes A dislikes D), yet Jaccard says 0.75 and raw cosine says 0.51. Only the centered version shows it.

### Questions and model answers

#### The professor's questions (Exercises)

**Q1. How does Jaccard differ from Cosine in CF regarding rating values? Which one ignores the magnitude?**
Jaccard treats ratings as binary (rated / not rated): it only compares the *sets* of users who interacted with the two items, $|U_i\cap U_j|/|U_i\cup U_j|$, so it **ignores the rating magnitudes** (1 star and 5 stars count the same). Cosine treats each item as a vector of rating values and measures the angle between the vectors, so the actual values matter.
*Extra credit:* raw cosine on strictly positive ratings cannot express disagreement (all products are positive); mean-centering (adjusted cosine / Pearson) fixes this. Jaccard is a natural fit for implicit feedback.

**Q2. Time complexity of building a full kNN map for $N$ items? Why is this an inefficiency?**
All pairwise similarities must be computed: $N(N-1)/2$ pairs → $O(N^2)$ similarity computations (each costs $O(\bar u)$, so $O(N^2\bar u)$ overall). Quadratic growth makes it infeasible for millions of items (10⁶ items → ~5·10¹¹ pairs), and the map must be rebuilt as data changes. Remedies: approximate NN search (LSH), GPU matrix multiplication, the Amazon item-to-item algorithm that skips pairs with no common user, or candidate sampling. Storage is only $O(Nk)$ because just the top-$k$ neighbours are kept.

**Q3. Difference between MAE and Hit Ratio; when is each appropriate?**
MAE measures the average absolute difference between predicted and true ratings — a *regression* error for the **rating prediction** task (lower is better). Hit Ratio measures whether a held-out relevant item appears in the top-N list — a *ranking* metric for the **Top-N recommendation** task (higher is better). MAE needs explicit ratings and treats all items equally; Hit Ratio only cares about the top of the list, which is what the user actually sees, and also works with implicit feedback.

**Q4.** → worked out step by step just above: Jaccard(A,D) = **0.75**; Cosine(A,D) = 18/√1260 = **0.507**; $\hat r_{U2,B}$ = 4.0/1.25 = **3.2**.

**Q5. Hit Ratio.** U1: hit (Movie 10 in list) · U2: miss · U3: hit · U4: hit · U5: miss → HR = 3/5 = **0.60**.

**In-text question — Cosine vs Jaccard result.** → full answer in 2.C. Short exam version:
> In the notebook, cosine is computed only over co-rated users and all ratings are positive, so almost every similarity is close to 1 (and exactly 1 when there is a single common user). The weights are nearly uniform, so the prediction degenerates to the user's average rating. Jaccard ignores rating values but captures how strongly two items co-occur; its values are spread out, so it actually discriminates between relevant and irrelevant neighbours. Hence Jaccard gives a lower MAE here. Mean-centering and significance weighting would fix cosine.

**In-text question — Efficiency: "Can you suggest a solution?"** → answer in 2.C (candidate generation + precomputed top-k similarities + ANN).

#### Other likely questions — algorithmic

**A1. Write the item-based rating prediction algorithm in pseudocode and give its complexity.** → `PREDICT` in 2.B; $O(|I_u|\cdot\bar u)$.

**A2. Write the Top-N algorithm using a kNN map. What are the offline and online parts?** → `TOPN_KNN_MAP` in 2.B. Offline: building the map, $O(n^2\bar u)$ time, $O(nk)$ space. Online: $O(|I_u|\cdot k)$ per request, independent of catalog size.

**A3. Compare "Predict & Sort" with the kNN-map approach.**
Predict & Sort predicts a rating for every unseen item ($O(n\,|I_u|\,\bar u)$ per request) and recomputes the same similarities for every user. The kNN-map approach precomputes similarities once, stores only the top-$k$ per item, and at request time only looks at neighbours of the user's items — orders of magnitude faster, at the cost of an offline build and approximate scores.

**A4. Explain the Amazon item-to-item algorithm and why it is faster on sparse data.** → pseudocode in 2.B.
It only computes similarities for item pairs that share at least one user, by iterating item → users who bought it → other items those users bought. In sparse data the vast majority of pairs have no common user, so they are skipped. Worst case $O(N^2M)$, in practice close to $O(NM)$ because most users have few interactions.

**A5. How do you keep the k most similar items efficiently while scanning candidates?**
Keep a bounded structure of size $k$ whose smallest element is cheap to access (sorted list as in the notebook, or a min-heap). For each candidate, if its similarity exceeds the current minimum, insert it and remove the minimum. Memory $O(k)$; with a heap the total time is $O(c\log k)$ for $c$ candidates instead of $O(c\log c)$ for a full sort.

**A6. Given `knn_map[X] = [(0.9,P),(0.5,Q)]`, `knn_map[Y] = [(0.6,Q),(0.4,R)]` and a user who watched X and Y, give the top-2 under (a) total similarity, (b) max similarity, (c) frequency.**
(a) Q=1.1, P=0.9 → [Q, P]. (b) P=0.9, Q=0.6 → [P, Q]. (c) Q=2, then P or R (tie) → [Q, P/R].

**A7. Compute the adjusted cosine between A and D in the Q4 matrix.** → worked out just above: **−0.943**. Interpretation: the items are strongly *dis*similar although Jaccard (0.75) and raw cosine (0.507) both suggest similarity.

**A8. Using the Q4 matrix, predict User 1's rating for Item C with Jaccard and all rated items.**
$U_C=\{2,4\}$. $J(C,A)=|\{2,4\}|/|\{1,2,3,4\}|=0.5$; $J(C,B)=|\{4\}|/|\{1,2,3,4\}|=0.25$; $J(C,D)=|\{2\}|/|\{1,2,3,4\}|=0.25$.
$\hat r = \dfrac{0.5\cdot5+0.25\cdot3+0.25\cdot1}{0.5+0.25+0.25} = \dfrac{3.5}{1.0} = 3.5$.

#### Other likely questions — conceptual

**B1. Why is mean-centering needed for cosine on explicit ratings?**
Ratings are strictly positive, so every term of the dot product is positive and disagreement (5 vs 1) still increases similarity — it even contributes more than agreeing on a low rating (1 vs 1). Subtracting the user's mean makes "liked" positive and "disliked" negative, so agreement adds and disagreement subtracts. It also removes user rating-scale bias (strict vs generous raters).

**B2. When is raw cosine appropriate?** Implicit binary feedback: $1\times1$ counts a co-occurrence and everything else is 0; there is no notion of a negative rating.

**B3. User-based vs item-based CF — why did industry prefer item-based?**
There are usually fewer items than users and item–item similarities are more stable over time, so they can be precomputed offline and reused; a user's vector changes with every interaction. Item-based recommendations are also easy to explain.

**B4. What is the effect of $k$?** Small $k$: noisy, high variance, low coverage. Large $k$: includes weakly related neighbours, predictions regress toward the user's mean. Tuned on a validation set.

**B5. Why are `rating_map` and `user_ratings_map` built?** To replace repeated DataFrame scans with $O(1)$ dictionary lookups inside the nested similarity/prediction loops.

**B6. What are the limitations of neighbourhood CF?** Cold start for new users/items; sparsity (few co-ratings → unreliable similarities); $O(n^2)$ offline cost; popularity bias; no use of content or context.

**B7. Is the Hit Ratio of Part 1 (1+99 negatives) comparable to the Hit Ratio in the kNN notebook?** No. The first ranks the positive among 100 sampled candidates; the second requires it to appear in a top-N list drawn from the entire catalog. Only numbers obtained under the same protocol are comparable.

**B8 (code reading). What is wrong with `add_sims_and_sort`?** `itertools.groupby` only groups *consecutive* equal keys, so the list must be sorted by movieId first; otherwise similarities of the same movie are not summed. Fix: sort by key before grouping, or accumulate in a dictionary.

---
# Part 3 — Matrix Factorization, BPR and LightGCN

This part covers `gnn.ipynb`. That notebook is almost only code, so the explanations here are longer.

## 3.A The real-world problem

**Situation.** StreamBox has grown, and two facts make the kNN approach of Part 2 struggle.

**Fact 1 — users do not rate, they just watch.** On real platforms only a few percent of users ever give a star rating. What you have plenty of is behaviour: plays, clicks, purchases, check-ins. This is called **implicit feedback**, and it is the normal case in industry (YouTube, Spotify, TikTok, any online shop). The dataset in the professor's notebook, **Gowalla**, is of this kind: a list of "user X checked in at place Y".

Implicit feedback is different from ratings in one important way:

| | Explicit (ratings) | Implicit (behaviour) |
|---|---|---|
| A filled cell means | "I rate this 4 stars" | "I interacted with this" |
| An empty cell means | "I have not rated it" | **unknown**: maybe the user dislikes it, maybe they never saw it |
| Negative examples | yes (1–2 stars) | **none** |
| Amount of data | small | very large |

So we have only positives and a huge number of unknowns. This changes three things:

- **MAE/RMSE make no sense** (there is no rating to predict) → we use ranking metrics: Precision@K, Recall@K, NDCG@K.
- **We need a loss function built for ranking** → BPR.
- **We must invent negatives** → negative sampling.

**Fact 2 — the catalog is huge and the data is sparse.** kNN has three limits:

1. **No common users → similarity 0.** Two niche movies that are in fact very similar, but were never watched by the same person, look unrelated. In sparse data this is the usual case.
2. **$O(n^2)$** work to compare all items, and a table of neighbours to store.
3. **Only one step.** kNN sees "A and B were watched by the same people". It does not see "A is like B, B is like C, so A is probably a bit like C".

**The new idea: learn a short vector for every user and every item.** Instead of storing similarities, describe each user and each item by a few numbers (an **embedding**), chosen so that a user's vector and an item's vector "match" when the user likes the item. Because every item lives in the same small space, two items that never shared a user can still end up close to each other.

**Where these methods are used:**

| Method | Real-world use |
|---|---|
| Matrix Factorization (ALS, BPR) | The workhorse for implicit data at many companies; produces the embeddings used for candidate retrieval |
| Graph neural networks (LightGCN and relatives) | When the "who interacted with what" graph carries extra signal and there is budget for training. Pinterest's PinSage is the best-known production example |

## 3.B Solving it by hand

### Step 1 — measuring a ranked list: Precision, Recall, NDCG

**Situation.** We show Ana a row of 5 movies, in this order: `[a, b, c, d, e]`. Later we learn which movies she actually went on to watch (the test set): `{b, e, z}`. How good was the row?

Mark each position: is the movie one she really watched?

| Position | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| Movie | a | **b** | c | d | **e** |
| Hit? | no | yes | no | no | yes |

**Precision@5 — "how much of what we showed was right?"**

$$\text{Precision@}K = \frac{\text{hits in the top } K}{K} = \frac{2}{5} = 0.40$$

**Recall@5 — "how much of what she wanted did we find?"** She wanted 3 movies (b, e, z). We found 2.

$$\text{Recall@}K = \frac{\text{hits in the top } K}{\text{number of relevant items}} = \frac{2}{3} = 0.67$$

**Why both?** They pull in opposite directions. Show 1,000 movies and recall is high but precision is terrible. Show one safe movie and precision is high but recall is low.

**The problem with both: they ignore the order.** The list `[b, e, a, c, d]` has the same precision and recall as ours, but it is clearly better, because the hits are at the top where the user looks first.

**NDCG@5 — rewards hits near the top.** Each hit earns a gain that shrinks with its position:

| Position $p$ | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| Gain of a hit $= 1/\log_2(p+1)$ | 1.000 | 0.631 | 0.500 | 0.431 | 0.387 |

1. **DCG** (Discounted Cumulative Gain) = add the gains of the hits. Our hits are at positions 2 and 5:
   $\text{DCG} = 0.631 + 0.387 = 1.018$
2. **IDCG** (Ideal DCG) = the DCG of the best possible list. She has 3 relevant movies, so the ideal list has hits at positions 1, 2, 3:
   $\text{IDCG} = 1.000 + 0.631 + 0.500 = 2.131$
3. **NDCG** = DCG / IDCG $= 1.018 / 2.131 = 0.478$

**Why divide by IDCG?** A user with 10 relevant movies can collect more gain than a user with 1. Dividing by the best achievable score puts every user on a 0-to-1 scale, so users can be averaged fairly. That is the "N" (normalised).

For comparison, the better list `[b, e, a, c, d]` has DCG $= 1.000 + 0.631 = 1.631$ and NDCG $= 0.765$. Same precision and recall, much higher NDCG.

**Link to Part 1:** if each user has exactly one test item, Recall@K is the same thing as Hit Ratio@K.

**Why "@K"?** Users only look at the first screen. Whether a relevant movie is at position 500 or 5,000 makes no difference to anybody.

In [31]:
# the professor's functions (unchanged)
def recall_at_k(actual, predicted, k):
    if len(actual) == 0: return 0.0
    return len(set(predicted[:k]) & set(actual)) / len(set(actual))

def precision_at_k(actual, predicted, k):
    if len(actual) == 0: return 0.0
    return len(set(predicted[:k]) & set(actual)) / k

def ndcg_at_k(actual, predicted, k):
    if len(actual) == 0: return 0.0
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(actual), k)))
    dcg  = sum(1.0 / math.log2(i + 2) for i, p in enumerate(predicted[:k]) if p in actual)
    return dcg / idcg

a_, p_ = ["b", "e", "z"], ["a", "b", "c", "d", "e"]
print(f"P@5={precision_at_k(a_, p_, 5):.3f}  R@5={recall_at_k(a_, p_, 5):.3f}  NDCG@5={ndcg_at_k(a_, p_, 5):.3f}")

P@5=0.400  R@5=0.667  NDCG@5=0.478


In the professor's code the positions are counted from 0, so the gain is written `1/log2(i+2)`. It is the same formula.

### Step 2 — Matrix Factorization: describing users and movies with a few numbers

**Idea.** Suppose every movie could be described by two numbers: *how much animation* and *how much action*. And every user by two numbers: *how much they like animation* and *how much they like action*. Then the predicted interest is

$$\text{score}(u, i) = (\text{user likes animation}) \times (\text{movie is animation}) + (\text{user likes action}) \times (\text{movie is action})$$

This is a **dot product** of two short vectors: $\hat x_{ui} = \mathbf p_u \cdot \mathbf q_i$.

**By hand.** Take these vectors, written as [animation, action]:

| | Vector |
|---|---|
| Ana | [2.0, 0.5] |
| Ben | [0.4, 2.1] |
| Toy Story | [2.2, 0.4] |
| Matrix | [0.2, 2.0] |
| Terminator | [0.1, 2.2] |

- Ana · Toy Story $= 2.0\times2.2 + 0.5\times0.4 = 4.4 + 0.2 = 4.6$ (she rated it 5)
- Ana · Matrix $= 2.0\times0.2 + 0.5\times2.0 = 0.4 + 1.0 = 1.4$ (she rated it 1)
- Ben · Matrix $= 0.4\times0.2 + 2.1\times2.0 = 0.08 + 4.2 = 4.3$ (he rated it 5)
- **Ana · Terminator** $= 2.0\times0.1 + 0.5\times2.2 = 0.2 + 1.1 = $ **1.3**

The model answers our running question correctly (low score), and for the right reason: Terminator has a large "action" number and Ana has a small one. Note that this prediction did **not** need any user who rated both Terminator and something else; it only needed the two vectors.

**Three things to understand about MF:**

1. **Nobody writes these vectors.** In a real model we do not choose "animation" and "action". The model starts from random numbers and adjusts them until the scores match the data. The dimensions it discovers are called **latent factors**; they often turn out to mean something (genre, seriousness, target age), but not always. Typical size: 32 to 256 numbers per user and per item.
2. **The name.** Stack all user vectors as the rows of a matrix $P$ and all item vectors as the rows of $Q$. Then the whole table of scores is $P\,Q^\top$. We are approximating the big rating matrix by the product of two thin matrices, i.e. *factorizing* it.
3. **Why it is compact.** 100 million users × 1 million items is $10^{14}$ cells. With 64 numbers per user and per item we store $(10^8 + 10^6)\times 64 \approx 6.5$ billion numbers. That fits; the full table never could.

In PyTorch, `nn.Embedding(num_users, 64)` is exactly the matrix $P$: a table with one trainable row per user.

### Step 3 — BPR: how the vectors are learned from implicit data

**The problem.** With ratings we could train by "make the predicted rating close to the true rating". With implicit data there is no rating. All we know is: *Ana watched Toy Story; she did not watch Terminator.*

**The BPR idea (Bayesian Personalized Ranking).** We do not need the score to be any particular number. We only need the **order** to be right:

> An item the user interacted with should get a higher score than an item they did not interact with.

So each training example is a **triple** $(u, i, j)$: a user $u$, an item $i$ they interacted with (the positive), and a randomly chosen item $j$ they did not interact with (the negative). Picking $j$ at random is called **negative sampling**.

**The loss for one triple:**

$$\text{loss} = -\ln \sigma(\hat x_{ui} - \hat x_{uj}) \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

Read it from the inside out:

1. $\hat x_{ui} - \hat x_{uj}$ is the **gap**: how far the positive is ahead of the negative.
2. $\sigma$ squeezes the gap into a number between 0 and 1: "how confident is the model that the order is right".
3. $-\ln$ turns high confidence into a small loss and low confidence into a large loss.

| Gap $\hat x_{ui} - \hat x_{uj}$ | $\sigma$(gap) | Loss | Meaning |
|---|---|---|---|
| +3 | 0.95 | 0.05 | order clearly right → almost nothing to fix |
| 0 | 0.50 | 0.69 | undecided (this is where training starts) |
| −3 | 0.05 | 3.05 | order clearly wrong → big correction |

**One training step by hand.** User Ana, positive $i$ = Toy Story, negative $j$ = Terminator. Start from poor vectors:

$\mathbf p_u = [1, 0]$, $\mathbf q_i = [0.5, 0.5]$, $\mathbf q_j = [1, 1]$, learning rate 0.1.

1. Scores: $\hat x_{ui} = 1\times0.5 + 0\times0.5 = 0.5$; $\hat x_{uj} = 1\times1 + 0\times1 = 1.0$. The negative is ahead. Wrong order.
2. Gap $= 0.5 - 1.0 = -0.5$. $\sigma(-0.5) = 0.378$. Loss $= -\ln 0.378 = 0.974$.
3. Size of the correction: $g = 1 - \sigma(\text{gap}) = 1 - 0.378 = 0.622$. (When the order is already right, $g$ is close to 0 and almost nothing changes.)
4. Move each vector a small step in the direction that widens the gap:
   - user: $\mathbf p_u \leftarrow \mathbf p_u + 0.1 \times g \times (\mathbf q_i - \mathbf q_j) = [1,0] + 0.0622\times[-0.5,-0.5] = [0.969, -0.031]$
   - positive item: $\mathbf q_i \leftarrow \mathbf q_i + 0.1 \times g \times \mathbf p_u = [0.5,0.5] + 0.0622\times[1,0] = [0.562, 0.5]$ — pulled **toward** the user
   - negative item: $\mathbf q_j \leftarrow \mathbf q_j - 0.1 \times g \times \mathbf p_u = [1,1] - 0.0622\times[1,0] = [0.938, 1]$ — pushed **away** from the user
5. New gap $= -0.348$, new loss $= 0.882$. Still wrong, but better. Repeat this millions of times over all triples and the vectors settle.

That is all "training" means here: for each observed interaction, pull the user and the item together and push a random other item away.

**The full algorithm:**

```
BPR_TRAIN(interactions, d, lr, epochs):
    P <- small random numbers (one row of d numbers per user)
    Q <- small random numbers (one row of d numbers per item)
    repeat `epochs` times:
        for each observed interaction (u, i):             # in practice in mini-batches
            j   <- a random item                          # negative sampling
            gap <- P[u].Q[i] - P[u].Q[j]
            g   <- 1 - sigmoid(gap)
            P[u] <- P[u] + lr * g * (Q[i] - Q[j])
            Q[i] <- Q[i] + lr * g * P[u]
            Q[j] <- Q[j] - lr * g * P[u]
    return P, Q

RECOMMEND(u, N):
    scores <- Q . P[u]                # one score per item
    remove the items u already has
    return the N items with the highest score
```

In the professor's code the three update lines are not written out. He computes the loss and calls `loss.backward()`; PyTorch derives the same updates automatically, and the Adam optimizer applies them.

**Why it is called a pairwise loss.** It compares *pairs* of items for the same user. A "pointwise" loss (such as squared error) looks at one item at a time. For ranking, pairwise is the more natural fit.

In [32]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
p_u, q_i, q_j, lr = np.array([1.0, 0.0]), np.array([0.5, 0.5]), np.array([1.0, 1.0]), 0.1   # i = positive, j = negative

x_uij = p_u @ q_i - p_u @ q_j
print(f"before: x_ui={p_u@q_i:.2f}  x_uj={p_u@q_j:.2f}  x_uij={x_uij:.2f}  loss={-np.log(sigmoid(x_uij)):.3f}   (negative is ranked ABOVE positive)")
g = 1 - sigmoid(x_uij)
p_new, qi_new, qj_new = p_u + lr * g * (q_i - q_j), q_i + lr * g * p_u, q_j - lr * g * p_u
x_new = p_new @ qi_new - p_new @ qj_new
print(f"g = 1 - sigmoid(x_uij) = {g:.3f}")
print(f"after : p_u={p_new.round(3)}  q_i={qi_new.round(3)}  q_j={qj_new.round(3)}")
print(f"        x_uij={x_new:.3f}  loss={-np.log(sigmoid(x_new)):.3f}   (gap moved in the right direction)")

before: x_ui=0.50  x_uj=1.00  x_uij=-0.50  loss=0.974   (negative is ranked ABOVE positive)
g = 1 - sigmoid(x_uij) = 0.622
after : p_u=[ 0.969 -0.031]  q_i=[0.562 0.5  ]  q_j=[0.938 1.   ]
        x_uij=-0.348  loss=0.882   (gap moved in the right direction)


### Step 4 — LightGCN: letting neighbours in the graph share information

**A different picture of the same data.** Draw every user and every item as a dot. Draw a line between a user and an item if the user interacted with it. This is a **bipartite graph** (two kinds of dots, lines only between the two kinds).

Tiny example: Ana watched Toy Story and Nemo. Cem watched Nemo and Titanic.

```
Ana ──── Toy Story
   \
    ──── Nemo
   /
Cem ──── Titanic
```

Ana and Cem are connected **through** Nemo. And Ana is connected to Titanic through the path Ana – Nemo – Cem – Titanic. That path is the collaborative-filtering signal: *someone who shares a movie with Ana also watched Titanic.*

**What plain MF does with this.** Ana's vector is only updated by Ana's own lines. Information from Cem reaches her only slowly and indirectly, over many training steps.

**What LightGCN does.** Before computing scores, it lets every dot **mix its vector with the vectors of its neighbours**, and repeats this a few times. Each round is called a **layer**, and the mixing is called **message passing**.

- After 1 layer, Ana's vector contains her movies (Toy Story, Nemo).
- After 2 layers, it contains the users who share those movies (Cem). This is the "similar users" idea of Part 2.
- After 3 layers, it contains those users' movies (Titanic), i.e. the candidates to recommend.

So **LightGCN puts the neighbourhood idea of kNN inside the embedding model.** That is how Parts 2 and 3 connect.

**By hand.** The interaction matrix (rows Ana, Cem; columns Toy Story, Nemo, Titanic):

$$R = \begin{bmatrix} 1 & 1 & 0 \\ 0 & 1 & 1 \end{bmatrix}$$

**(a) Degrees** = number of lines at each dot: Ana 2, Cem 2, Toy Story 1, Nemo 2, Titanic 1.

**(b) Weight of each line** $= \dfrac{1}{\sqrt{\deg(\text{one end}) \times \deg(\text{other end})}}$:

| Line | Calculation | Weight |
|---|---|---|
| Ana – Toy Story | $1/\sqrt{2\times1}$ | 0.707 |
| Ana – Nemo | $1/\sqrt{2\times2}$ | 0.500 |
| Cem – Nemo | $1/\sqrt{2\times2}$ | 0.500 |
| Cem – Titanic | $1/\sqrt{2\times1}$ | 0.707 |

**Why these weights?** Without them, a dot with 10,000 neighbours would add up 10,000 vectors and become enormous, while a dot with 2 neighbours stays small. Training would become unstable and popular items would dominate everything. Dividing by the degrees of both ends keeps all vectors on a similar scale. It is called *symmetric normalisation*, written $\hat A = D^{-1/2} A D^{-1/2}$, and it is what the professor's `d_inv` computes.

**(c) Starting vectors** (layer 0, chosen for easy arithmetic):

Ana [1, 0] · Cem [0, 1] · Toy Story [1, 1] · Nemo [2, 0] · Titanic [0, 2]

**(d) Layer 1: each dot becomes the weighted sum of its neighbours.**

- Ana $= 0.707\times\text{Toy Story} + 0.5\times\text{Nemo} = 0.707\times[1,1] + 0.5\times[2,0] = [1.707,\ 0.707]$
- Cem $= 0.5\times[2,0] + 0.707\times[0,2] = [1.0,\ 1.414]$
- Toy Story $= 0.707\times\text{Ana} = [0.707,\ 0]$
- Nemo $= 0.5\times\text{Ana} + 0.5\times\text{Cem} = [0.5,\ 0.5]$
- Titanic $= 0.707\times\text{Cem} = [0,\ 0.707]$

**(e) Layer 2: the same rule, applied to the layer-1 vectors.**

- Ana $= 0.707\times[0.707, 0] + 0.5\times[0.5, 0.5] = [0.75,\ 0.25]$
- Titanic $= 0.707\times[1.0, 1.414] = [0.707,\ 1.0]$

**(f) Final vector = the average of all layers** (layer 0, 1 and 2):

- Ana $= \big([1,0] + [1.707, 0.707] + [0.75, 0.25]\big)/3 = [1.152,\ 0.319]$
- Titanic $= \big([0,2] + [0, 0.707] + [0.707, 1.0]\big)/3 = [0.236,\ 1.236]$

**(g) The score, before and after.**

- Plain MF with the starting vectors: Ana · Titanic $= 1\times0 + 0\times2 = $ **0**
- LightGCN: Ana · Titanic $= 1.152\times0.236 + 0.319\times1.236 = 0.272 + 0.394 = $ **0.67**

Ana never touched Titanic, and the starting vectors said "no relation at all". After two rounds of mixing, the graph has carried Cem's information to Ana, and Titanic has a positive score.

**Why average the layers** instead of using only the last one?

1. **Over-smoothing.** If you keep mixing, every dot's vector becomes a blend of everybody and all vectors look alike. Then the model cannot tell users apart. Keeping the early layers in the average preserves each dot's own identity.
2. Different layers carry different information (own identity, own items, similar users, their items). The average keeps all of it.

**Why "Light"?** Ordinary graph neural networks multiply by a trainable weight matrix and apply a non-linear function in every layer. The LightGCN paper showed that for recommendation these two steps do not help and make training harder, so it removes them. What remains is just the mixing rule above.

**How it relates to MF:**

- The **trainable numbers are exactly the same** as in MF: one starting vector per user and per item. The mixing step has no parameters.
- The **loss is the same** (BPR).
- With **0 layers, LightGCN is exactly MF.**

**The algorithm:**

```
LIGHTGCN_FORWARD(E0, A_hat, K):          # E0: one starting vector per user and per item (the only trainable numbers)
    E <- E0 ;  layers <- [E0]
    for k = 1..K:
        E <- A_hat @ E                   # one round of message passing = one sparse matrix product
        layers.append(E)
    E_final <- average of layers
    split E_final into user vectors and item vectors
    score(u, i) <- user vector . item vector      # then train with BPR, exactly like MF
```

**Cost.** One round touches every line of the graph once: $O(|E| \cdot d)$ for $|E|$ interactions and vectors of size $d$; $K$ rounds cost $K$ times that. MF needs only the vectors in the current batch. That is why each LightGCN epoch is slower than an MF epoch in the professor's output.

In [33]:
# The LightGCN hand example, checked: Ana and Cem; Toy Story, Nemo, Titanic
R_toy = np.array([[1, 1, 0],      # Ana -> Toy Story, Nemo
                  [0, 1, 1]])     # Cem -> Nemo, Titanic
names = ["Ana", "Cem", "ToyStory", "Nemo", "Titanic"]
A = np.block([[np.zeros((2, 2)), R_toy], [R_toy.T, np.zeros((3, 3))]])     # adjacency of the whole graph
deg = A.sum(1)
A_hat = np.diag(deg ** -0.5) @ A @ np.diag(deg ** -0.5)                     # symmetric normalisation
print("degrees:", dict(zip(names, deg.astype(int).tolist())))
print(pd.DataFrame(A_hat.round(3), index=names, columns=names), "\n")

E0 = np.array([[1, 0], [0, 1], [1, 1], [2, 0], [0, 2.0]])
E1 = A_hat @ E0
E2 = A_hat @ E1
E_final = (E0 + E1 + E2) / 3
print(pd.DataFrame(np.hstack([E0, E1, E2, E_final]).round(3), index=names,
                   columns=["L0_a", "L0_b", "L1_a", "L1_b", "L2_a", "L2_b", "final_a", "final_b"]), "\n")
print("score Ana . Titanic   plain MF (layer 0 only):", E0[0] @ E0[4], "   LightGCN:", round(float(E_final[0] @ E_final[4]), 3))

degrees: {'Ana': 2, 'Cem': 2, 'ToyStory': 1, 'Nemo': 2, 'Titanic': 1}
            Ana    Cem  ToyStory  Nemo  Titanic
Ana       0.000  0.000     0.707   0.5    0.000
Cem       0.000  0.000     0.000   0.5    0.707
ToyStory  0.707  0.000     0.000   0.0    0.000
Nemo      0.500  0.500     0.000   0.0    0.000
Titanic   0.000  0.707     0.000   0.0    0.000 

          L0_a  L0_b   L1_a   L1_b   L2_a   L2_b  final_a  final_b
Ana        1.0   0.0  1.707  0.707  0.750  0.250    1.152    0.319
Cem        0.0   1.0  1.000  1.414  0.250  0.750    0.417    1.055
ToyStory   1.0   1.0  0.707  0.000  1.207  0.500    0.971    0.500
Nemo       2.0   0.0  0.500  0.500  1.354  1.061    1.285    0.520
Titanic    0.0   2.0  0.000  0.707  0.707  1.000    0.236    1.236 

score Ana . Titanic   plain MF (layer 0 only): 0.0    LightGCN: 0.666


## 3.C Professor's code — what it does

**Pipeline of the `gnn.ipynb` notebook:**

1. **Data:** Gowalla → only the (user, item) columns → **50-core** filter (users and items with at least 50 interactions, for speed) → map the IDs to contiguous indices `0..N-1` (required by `nn.Embedding`).
2. **Split:** for each user 80% train / 20% test, at random.
3. **`R`:** the binary sparse train matrix (user × item).
4. **Metrics:** the three functions above + `evaluate_model`: it gets the embeddings **once**; for each user `scores = u_emb @ items_emb.T`; sets the train items to `-inf`; `torch.topk`.
5. **`MatrixFactorization`:** two `nn.Embedding`s; `forward(users, pos, neg)` → `(u*pos).sum(1)` and `(u*neg).sum(1)`.
6. **`LightGCN`:** the same two embeddings + `_build_norm_adj` (builds $\hat A$) + a loop of `torch.sparse.mm` + `torch.mean` over the layers.
7. **`BPRDataset`:** each sample is `(user, pos_item, neg_item)` with `neg_item = randint(num_items)`.
8. **Train:** Adam, `loss = -F.logsigmoid(pos_scores - neg_scores).mean()`.

**The key lines of the professor's LightGCN:**

```python
# A = [[0, R], [R^T, 0]]
adj_mat[:num_users, num_users:] = R
adj_mat[num_users:, :num_users] = R.T
# D^{-1/2}
rowsum = np.array(adj_mat.sum(axis=1))
d_inv  = np.power(rowsum, -0.5).flatten()
d_inv[np.isinf(d_inv)] = 0.                        # isolated nodes: degree 0 -> inf -> set to 0
norm_adj = d_mat.dot(adj_mat).dot(d_mat)           # D^{-1/2} A D^{-1/2}

# forward
all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight])   # E^(0): users stacked on items
embs = [all_emb]
for layer in range(self.n_layers):
    all_emb = torch.sparse.mm(norm_adj, all_emb)   # E^(k+1) = A_hat E^(k)
    embs.append(all_emb)
light_out = torch.mean(torch.stack(embs, dim=1), dim=1)             # average over layers
users_emb, items_emb = torch.split(light_out, [num_users, num_items])
```

**Flaws and notes:**

1. **Leakage / duplicates in the data:** Gowalla is check-ins, and a user can visit the same place several times. The split is done on check-ins, not on unique pairs; it is visible in the professor's own output (`User 0: [32, 27, 1, 1, 46]`). So the same (user, item) pair can be in both train and test. And because the train items are masked with `-inf` during evaluation, those test items can **never** be hit → recall is artificially lowered. Fix: `drop_duplicates(['user','item'])` before the split.
2. **Simple negative sampling:** `randint` does not check that the item is really a negative (false negatives are possible). A common and cheap approximation, because the data is very sparse.
3. **No L2 regularization** — the LightGCN and BPR papers include the term $\lambda\lVert\Theta\rVert^2$.
4. **The propagation logic is copied twice** (`forward` and `get_embeddings`); the name `g_droped_adj` is left over from the original code, but there is no dropout.
5. **Building the adjacency with `dok_matrix` and a loop** is slow; `scipy.sparse.bmat` does it in one line.
6. **Only 10 epochs:** both models are under-trained (Recall@20 for MF ≈ 0.09), so the final comparison there is not precise.

### Why this code is fine for a class but not how it is done at work

| What the professor does | Why it is a problem in practice | What is done instead |
|---|---|---|
| Writes MF, LightGCN, BPR and the metrics from scratch | Good for learning; at work it is slow to write, easy to get subtly wrong, and not optimised | Tested libraries: `implicit` for MF, PyTorch Geometric or RecBole for LightGCN |
| Builds the matrix cell by cell in a Python loop (`dok_matrix`) | Very slow for millions of interactions | Build the sparse matrix in one call from the arrays of user and item indices |
| Random negatives, without checking that they are real negatives | Some "negatives" are items the user did interact with; and random negatives are mostly too easy | Check against the user's history; often sample harder or popularity-weighted negatives |
| A random 80/20 split | In production you predict the *future* from the *past*; a random split lets the model see the future | Split by time: train on earlier interactions, test on later ones |
| Evaluates one user at a time in a Python loop | Slow | Score all users in batches on the GPU |
| Scores every item for every user by a full dot product | Fine for 12,000 items, impossible online for millions | Put the item vectors in an approximate nearest-neighbour index (FAISS) and query it |
| No regularisation, 10 epochs, no validation set | The models are under-trained and untuned, so the comparison is not reliable | Tune on a validation set with early stopping |
| Split done on check-ins, not on unique (user, item) pairs | The same pair is in train and test (leakage), and those test items can never be recommended because train items are masked | Remove duplicates before splitting |

## 3.D The industry version, step 1: the same models on MovieLens

Instead of Gowalla (100MB) we turn the same MovieLens data into implicit feedback: `rating >= 4` means positive. This way the three parts of the notebook are comparable on one dataset. The split follows the professor: 80/20 per user (on unique pairs).

In [34]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(SEED)

pos_df = ratings[ratings.rating >= 4.0]
rng = np.random.default_rng(SEED)
train_pairs, test_data = [], {}
for u, g in pos_df.groupby("userId"):
    items = rng.permutation(i2idx[g.movieId].to_numpy())
    cut = int(0.8 * len(items)) if len(items) >= 5 else len(items)
    train_pairs += [(int(u2idx[u]), int(i)) for i in items[:cut]]
    if cut < len(items): test_data[int(u2idx[u])] = items[cut:].tolist()
train_pairs = np.array(train_pairs)
R = csr_matrix((np.ones(len(train_pairs)), (train_pairs[:, 0], train_pairs[:, 1])), shape=(n_users, n_items))
print(f"train interactions: {len(train_pairs)}   test users: {len(test_data)}")

def evaluate_scores(scores, k=20):
    """scores: users x items. Same logic as the professor's evaluate_model: mask train items, take top-k, average the metrics."""
    sc = np.array(scores, dtype=float)
    sc[R.nonzero()] = -np.inf
    top = np.argsort(-sc, axis=1)[:, :k]
    res = [(precision_at_k(a, top[u].tolist(), k), recall_at_k(a, top[u].tolist(), k), ndcg_at_k(set(a), top[u].tolist(), k)) for u, a in test_data.items()]
    return dict(zip([f"Precision@{k}", f"Recall@{k}", f"NDCG@{k}"], np.mean(res, axis=0)))

results3 = {"Popularity": evaluate_scores(np.tile(np.asarray(R.sum(0)).ravel(), (n_users, 1)))}
S3 = topk_item_sim(R, k=30, metric="cosine")
results3["item-kNN cosine (Part 2)"] = evaluate_scores((R @ S3).toarray())
pd.DataFrame(results3).T

train interactions: 38620   test users: 603


,Precision@20,Recall@20,NDCG@20
Popularity,0.0866,0.1450,0.1520
item-kNN cosine (Part 2),0.1191,0.2052,0.2075


### MF and LightGCN in PyTorch (the professor's structure, more compact)

Changes from the professor's code: the propagation is written in one place only (`get_embeddings`) and LightGCN inherits from MF — which itself shows that **this one function is the only difference between the two models**.

In [35]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)    # P: one trainable row per user
        self.item_emb = nn.Embedding(num_items, embedding_dim)    # Q: one trainable row per item
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def get_embeddings(self):
        return self.user_emb.weight, self.item_emb.weight

    def forward(self, users, pos_items, neg_items):
        U, I = self.get_embeddings()
        u = U[users]
        return (u * I[pos_items]).sum(1), (u * I[neg_items]).sum(1)   # x_ui, x_uj


def build_norm_adj(R):
    """A_hat = D^-1/2 [[0, R], [R^T, 0]] D^-1/2  as a sparse torch tensor."""
    A = sp.bmat([[None, R], [R.T, None]], format="csr")
    deg = np.asarray(A.sum(1)).ravel()
    with np.errstate(divide="ignore"):
        d_inv = np.power(deg, -0.5)
    d_inv[np.isinf(d_inv)] = 0.0                                  # isolated nodes
    A = (sp.diags(d_inv) @ A @ sp.diags(d_inv)).tocoo()
    return torch.sparse_coo_tensor(np.vstack([A.row, A.col]), A.data.astype(np.float32), A.shape).coalesce()


class LightGCN(MatrixFactorization):
    def __init__(self, num_users, num_items, R, embedding_dim=64, n_layers=2):
        super().__init__(num_users, num_items, embedding_dim)
        self.num_users, self.num_items, self.n_layers = num_users, num_items, n_layers
        self.norm_adj = build_norm_adj(R)

    def get_embeddings(self):
        E = torch.cat([self.user_emb.weight, self.item_emb.weight])   # E^(0)
        layers = [E]
        for _ in range(self.n_layers):
            E = torch.sparse.mm(self.norm_adj, E)                     # message passing
            layers.append(E)
        E = torch.stack(layers, dim=1).mean(dim=1)                    # layer combination
        return torch.split(E, [self.num_users, self.num_items])


def train_bpr(model, epochs=60, lr=5e-3, batch_size=2048):
    torch.manual_seed(SEED)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    U, I = torch.tensor(train_pairs[:, 0]), torch.tensor(train_pairs[:, 1])
    t0 = time.time()
    for ep in range(epochs):
        perm = torch.randperm(len(U))
        neg  = torch.randint(0, n_items, (len(U),))                   # negative sampling: fresh negatives every epoch
        tot  = 0.0
        for s in range(0, len(U), batch_size):
            b = perm[s:s + batch_size]
            pos_scores, neg_scores = model(U[b], I[b], neg[b])
            loss = -F.logsigmoid(pos_scores - neg_scores).mean()      # BPR loss
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        if (ep + 1) % 20 == 0:
            print(f"  {model.__class__.__name__:20s} epoch {ep+1:3d}  loss={tot/len(U):.4f}  ({time.time()-t0:.0f}s)")
    with torch.no_grad():
        Ue, Ie = model.get_embeddings()
        return (Ue @ Ie.T).numpy()

results3["MF + BPR (PyTorch)"]       = evaluate_scores(train_bpr(MatrixFactorization(n_users, n_items)))
results3["LightGCN + BPR (PyTorch)"] = evaluate_scores(train_bpr(LightGCN(n_users, n_items, R, n_layers=2)))
pd.DataFrame(results3).T

  MatrixFactorization  epoch  20  loss=0.0608  (4s)


  MatrixFactorization  epoch  40  loss=0.0393  (6s)


  MatrixFactorization  epoch  60  loss=0.0339  (8s)


  LightGCN             epoch  20  loss=0.0927  (11s)


  LightGCN             epoch  40  loss=0.0592  (22s)


  LightGCN             epoch  60  loss=0.0448  (33s)


,Precision@20,Recall@20,NDCG@20
Popularity,0.0866,0.1450,0.1520
item-kNN cosine (Part 2),0.1191,0.2052,0.2075
MF + BPR (PyTorch),0.1010,0.1860,0.1700
LightGCN + BPR (PyTorch),0.1290,0.2251,0.2219


**Reading the table:** with identical parameters and an identical loss, just adding message passing to MF raises the numbers; and both learned models are better than popularity. The price is a longer training time (compare the seconds in the log).

Look at the simple item-kNN too: a method with no training that is fully competitive. This result is well known in the literature, and it is the reason why in industry simple, **well-tuned** baselines are always kept next to the deep model.

## 3.E The industry version, step 2: libraries

### `implicit`: ALS and BPR

- **ALS (Alternating Least Squares)** — the most common MF for implicit data in industry (Spark MLlib has it too). Instead of SGD, it fixes $P$ and solves for $Q$ by least squares, then the other way round; it parallelizes easily and needs no negative sampling (it treats all empty cells as negatives with a low weight).
- **BPR** — the same algorithm as above, implemented in C++.

In [36]:
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking

als = AlternatingLeastSquares(factors=64, regularization=0.05, iterations=20, random_state=SEED)
als.fit(R, show_progress=False)
results3["implicit ALS"] = evaluate_scores(als.user_factors @ als.item_factors.T)

bpr = BayesianPersonalizedRanking(factors=64, learning_rate=0.01, regularization=0.01, iterations=100, random_state=SEED)   # library defaults
bpr.fit(R, show_progress=False)
results3["implicit BPR"] = evaluate_scores(bpr.user_factors @ bpr.item_factors.T)
pd.DataFrame(results3).T

,Precision@20,Recall@20,NDCG@20
Popularity,0.0866,0.1450,0.1520
item-kNN cosine (Part 2),0.1191,0.2052,0.2075
MF + BPR (PyTorch),0.1010,0.1860,0.1700
LightGCN + BPR (PyTorch),0.1290,0.2251,0.2219
implicit ALS,0.1150,0.2278,0.2216
implicit BPR,0.1045,0.1999,0.1930


The numbers are sensitive to the hyperparameters (factors, learning rate, regularization, number of iterations) and they are not tuned here. An example: when I raised the number of iterations and lowered the regularization for `implicit` BPR, it overfitted and Recall@20 fell from about 0.20 to below popularity. In real work you tune these on a separate validation set, not on the test set.

### LightGCN with PyTorch Geometric

PyG is the standard GNN library and ships a ready-made `LightGCN` (including the BPR loss with regularization). The nodes share one index space: users are `0..m-1`, items are `m..m+n-1`.

In [37]:
try:
    from torch_geometric.nn import LightGCN as PyGLightGCN

    torch.manual_seed(SEED)
    ei = torch.tensor(np.vstack([train_pairs[:, 0], train_pairs[:, 1] + n_users]))    # user -> item edges
    edge_index = torch.cat([ei, ei.flip(0)], dim=1)                                    # undirected: add both directions
    pyg = PyGLightGCN(num_nodes=n_users + n_items, embedding_dim=64, num_layers=2)
    opt = torch.optim.Adam(pyg.parameters(), lr=5e-3)

    for ep in range(60):
        perm = torch.randperm(ei.shape[1])
        neg  = torch.randint(n_users, n_users + n_items, (ei.shape[1],))
        for s in range(0, ei.shape[1], 2048):
            b = perm[s:s + 2048]
            pairs = torch.cat([ei[:, b], torch.stack([ei[0, b], neg[b]])], dim=1)      # positive pairs, then negative pairs
            pos_rank, neg_rank = pyg(edge_index, pairs).chunk(2)
            loss = pyg.recommendation_loss(pos_rank, neg_rank, node_id=pairs.unique(), lambda_reg=1e-4)   # BPR + L2
            opt.zero_grad(); loss.backward(); opt.step()

    with torch.no_grad():
        E = pyg.get_embedding(edge_index)
        results3["LightGCN (PyTorch Geometric)"] = evaluate_scores((E[:n_users] @ E[n_users:].T).numpy())
except ImportError:
    print("torch_geometric is not installed ->  %pip install torch_geometric")
pd.DataFrame(results3).T

,Precision@20,Recall@20,NDCG@20
Popularity,0.0866,0.1450,0.1520
item-kNN cosine (Part 2),0.1191,0.2052,0.2075
MF + BPR (PyTorch),0.1010,0.1860,0.1700
LightGCN + BPR (PyTorch),0.1290,0.2251,0.2219
implicit ALS,0.1150,0.2278,0.2216
implicit BPR,0.1045,0.1999,0.1930
LightGCN (PyTorch Geometric),0.1250,0.2184,0.2159


### Other tools whose names you should know

| Tool | Use |
|---|---|
| **RecBole** | dozens of models (BPR, LightGCN, …) driven by a config, for fair and fast comparison; mostly for research |
| **Spark MLlib ALS** | MF on distributed data |
| **TorchRec** (Meta), **TensorFlow Recommenders** (Google), **NVIDIA Merlin** | deep models at large scale |
| **FAISS / ScaNN** | serving: finding the item embeddings nearest to a user embedding |

**Common production architecture:** embeddings are trained offline (MF / two-tower / GNN) → item embeddings go into an ANN index → online: user embedding → a few hundred candidates from the index → a heavier ranker → business rules → display.

## 3.F Exam — MF / BPR / LightGCN

> `gnn.ipynb` contains no exercise questions; these are my predictions of likely questions.

**C1. What is the difference between explicit and implicit feedback, and how does it change the evaluation?**
Explicit feedback is a stated preference (ratings); implicit feedback is observed behaviour (clicks, check-ins, purchases) — only positives are observed and a missing entry is unknown, not negative. With implicit data rating-error metrics are meaningless, so ranking metrics (Precision@K, Recall@K, NDCG@K, Hit Ratio) are used, and training needs negative sampling.

**C2 (calculation). predicted = [a, b, c, d, e], actual = {b, e, z}. Compute Precision@5, Recall@5, NDCG@5.** → 3.B: **0.4, 0.667, 0.478**.

**C3. Why use NDCG in addition to Precision/Recall?**
Precision@K and Recall@K ignore the order inside the top-K. NDCG discounts each hit by $1/\log_2(\text{position}+1)$, so relevant items near the top count more; dividing by the ideal DCG normalises it to [0, 1].

**C4. Explain Matrix Factorization for recommendation.**
Each user and item is represented by a $d$-dimensional latent vector; the predicted preference is their dot product, $\hat x_{ui}=\mathbf p_u^\top\mathbf q_i$. The vectors are learned by minimising a loss on observed interactions. It compresses the $m\times n$ matrix into $(m+n)d$ parameters and generalises to pairs with no co-occurrence.

**C5. Write the BPR loss and explain the intuition. What is a training sample?**
$L=-\sum_{(u,i,j)}\ln\sigma(\hat x_{ui}-\hat x_{uj})$. A sample is a triple (user, positive item, sampled negative item). The loss is small when the positive is scored higher than the negative, so the model learns a correct *relative order* rather than absolute scores (pairwise ranking loss).

**C6 (calculation). $\hat x_{ui}=2.0$, $\hat x_{uj}=0.5$. Compute the BPR loss for this triple.**
$\sigma(1.5)=0.8176$ → loss $=-\ln 0.8176 = 0.201$. (If the scores were swapped: $\sigma(-1.5)=0.1824$ → loss $=1.70$.)

**C7. Describe LightGCN's propagation rule. How does it differ from MF and from a standard GCN?**
$E^{(k+1)}=D^{-1/2}AD^{-1/2}E^{(k)}$ on the user–item bipartite graph; the final embedding is the mean of the embeddings from all layers; the score is a dot product trained with BPR. Versus a standard GCN: no feature transformation matrices and no non-linear activation. Versus MF: same trainable parameters (only the layer-0 embeddings), but embeddings are smoothed over the graph so higher-order collaborative signals are encoded; with 0 layers it reduces to MF.

**C8. Why is the adjacency matrix symmetrically normalised? Compute the weight of an edge between nodes of degree 3 and 2.**
Without normalisation, high-degree nodes accumulate very large embeddings, which causes numerical instability; the factor $1/\sqrt{\deg(i)\deg(j)}$ keeps the scale consistent. Weight $=1/\sqrt{6}=0.408$.

**C9. What information does a user embedding contain after 1 / 2 / 3 layers?**
1: the items the user interacted with. 2: other users who share those items. 3: the items of those similar users (candidate recommendations).

**C10. What is over-smoothing and how does LightGCN mitigate it?**
With many layers, repeated neighbourhood averaging makes all node embeddings similar and they lose discriminative power. LightGCN uses few layers and averages the embeddings of all layers (including layer 0).

**C11. Why are training items masked (set to −∞) during evaluation?**
The model is trained to score them highly, so they would fill the top-K; but recommending already-seen items is useless and they are not in the test set, so they must be excluded.

**C12. Why is negative sampling needed and what is its weakness in the notebook?**
Implicit data has only positives; BPR needs an item assumed to be less preferred. The notebook samples a uniformly random item without checking whether the user interacted with it, so some "negatives" are actually positives (false negatives), and uniform negatives are mostly easy.

**C13. Build the adjacency matrix $A$ for $R=\begin{bmatrix}1&1&0\\0&1&1\end{bmatrix}$ and give the node degrees.**
$A=\begin{bmatrix}0&R\\R^\top&0\end{bmatrix}$ is $5\times5$; degrees: $u_0=2,\ u_1=2,\ i_0=1,\ i_1=2,\ i_2=1$. E.g. $\hat A_{u_0,i_0}=1/\sqrt{2\cdot1}=0.707$, $\hat A_{u_0,i_1}=1/\sqrt{2\cdot2}=0.5$.

---
# 4. Summary and night-before-the-exam review

### The whole week in one table

| | Baselines | Item-kNN CF | MF + BPR | LightGCN |
|---|---|---|---|---|
| Personalized | ✗ | ✓ | ✓ | ✓ |
| Training | none | none (precompute) | SGD / ALS | SGD |
| What is stored | $O(n)$ numbers | $O(nk)$ neighbours | $(m+n)d$ | $(m+n)d$ + graph |
| Offline cost | $O(\lvert R\rvert)$ | $O(n^2\bar u)$ | $O(\text{epochs}\cdot\lvert R\rvert d)$ | $O(\text{steps}\cdot K\lvert E\rvert d)$ |
| Online cost / request | $O(1)$ | $O(\lvert I_u\rvert k)$ | $O(nd)$ or ANN | $O(nd)$ or ANN |
| Explainability | high | high | low | low |
| Main weakness | same list for everyone | sparsity, $n^2$ | direct interactions only | training cost, over-smoothing |
| Library | pandas, Surprise | `implicit`, Surprise, FAISS | `implicit`, Spark ALS | PyG, RecBole |

### Formulas to know by heart

- MAE $=\frac1{|T|}\sum|r-\hat r|$ , RMSE $=\sqrt{\frac1{|T|}\sum(r-\hat r)^2}$ , Hit Ratio $=\frac{\#hits}{|T|}$
- Jaccard $=\frac{|A\cap B|}{|A\cup B|}$ , Cosine $=\frac{\mathbf u\cdot\mathbf v}{\lVert\mathbf u\rVert\lVert\mathbf v\rVert}$ , Adjusted cosine = cosine after subtracting $\bar r_u$
- Item-kNN: $\hat r_{ui}=\frac{\sum_j s_{ij}r_{uj}}{\sum_j|s_{ij}|}$
- Precision@K $=\frac{hits}{K}$ , Recall@K $=\frac{hits}{|actual|}$ , NDCG $=\frac{DCG}{IDCG}$ with $\frac{1}{\log_2(pos+1)}$
- MF: $\hat x_{ui}=\mathbf p_u^\top\mathbf q_i$ , BPR: $-\ln\sigma(\hat x_{ui}-\hat x_{uj})$
- LightGCN: $E^{(k+1)}=D^{-1/2}AD^{-1/2}E^{(k)}$ , $E=\text{mean}_k E^{(k)}$

### Reference numbers

- Random HR@K with 1+99 $= K/100$
- $1/\log_2$: rank 1 → 1, rank 2 → 0.631, rank 3 → 0.5, rank 4 → 0.431, rank 5 → 0.387
- $\sigma(0)=0.5$ , $-\ln 0.5 = 0.693$ (the BPR loss at the start of training, when the model knows nothing yet)
- kNN-map: time $O(n^2)$ , memory $O(nk)$

### Common exam mistakes

1. In a hand-computed cosine, take the vectors over the **common users** only (the professor's method), but in Jaccard the union is over **all** users of both items.
2. The denominator of the prediction is the **sum of the similarities**, not the number of neighbours.
3. The denominator of Recall is the number of **actual** items; the denominator of Precision is **K** itself.
4. Compute IDCG with $\min(K, |actual|)$ terms.
5. Lower MAE is better; higher HR/Precision/Recall/NDCG is better.
6. "Jaccard ignores the rating values" ≠ "Jaccard is worse" — in the professor's notebook it came out better, and you must be able to say why.

### To make the algorithms really stick (suggested practice)

- On the Q4 matrix, **without looking**, compute the Jaccard and cosine of every pair and check them with the functions of section 2.B.
- Write the pseudocode of these five algorithms from memory: HR evaluation, `PREDICT`, `TOPN_KNN_MAP`, Amazon item-to-item, `LIGHTGCN_FORWARD`.
- In the code of 2.D change `k` (5, 30, 200) and watch its effect on MAE and HR.
- In `LightGCN` set `n_layers` to 0, 1, 3; with 0 you should get exactly MF.